Manual expert annotation of all 12 clusters based on marker genes + dot plot evidence

In [1]:

import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os

os.makedirs('/mnt/results/05_annotation', exist_ok=True)

# ─── Manual annotation based on marker evidence ───────────────────────────────
# Cluster 0: COL9A3, SCRG1, ACAN, SERPINA5, OGN  → NP chondrocyte (ECM-rich, ACAN+)
# Cluster 1: FGFBP2, FN1, TIMP3, HAPLN1, EMP3    → NP chondrocyte (HAPLN1+, FN1+, TIMP3 = matrix-remodeling NP)
# Cluster 2: GADD45B, DNAJB1, JUN, SERTAD1, FOS  → Stress-response NP (heat shock / AP-1 stress)
# Cluster 3: EIF1, SLC3A2, SQSTM1, TNFRSF12A, DNAJB9 → Degenerative NP (UPR/autophagy: SQSTM1, DNAJB9)
# Cluster 4: COL1A2, COL3A1, COL1A1, COL6A3      → AF fibroblast (fibrocartilage)
# Cluster 5: MALAT1, MT1E, MT1G, MT1X             → Metallothionein+ NP (oxidative stress / senescent)
# Cluster 6: B2M, CXCR4, RPS29, MT-ND3           → T/NK cell (CXCR4+, B2M+, mitochondrial)
# Cluster 7: CALD1, TAGLN, MYL9, SPARCL1, NR2F2  → Pericyte/smooth muscle (TAGLN, MYL9, CALD1)
# Cluster 8: B2M, TMSB10, SPARCL1, IFI27         → Endothelial (SPARCL1+, IFI27+, B2M+)
# Cluster 9: HBB, HBA1, HBA2, AHSP, HBD          → Erythrocyte (hemoglobin genes)
# Cluster 10: SRGN, CXCL8, TYROBP, CTSS          → Macrophage/monocyte (TYROBP, CTSS, CXCL8)
# Cluster 11: LYZ, S100A9, S100A8, MNDA           → Neutrophil/monocyte (S100A8/A9, LYZ, MNDA)

# Cross-check with dot plot:
# - Clusters 0,1,2,3,5 all show ACAN+/COL2A1+/SOX9+ → NP lineage confirmed
# - Cluster 4: COL1A1+, SCX+, PDGFRA+ → AF fibroblast confirmed
# - Cluster 7: PECAM1 low, but TAGLN/MYL9 → pericyte/SMC
# - Cluster 8: PECAM1+, VWF+, CDH5+ → Endothelial confirmed
# - Cluster 10: CD68+, CD14+ → Macrophage confirmed
# - Cluster 11: CD68+, PTPRC+ → Monocyte/neutrophil

cluster_annotation = {
    '0': 'NP_chondrocyte',
    '1': 'NP_chondrocyte_HAPLN1',   # HAPLN1-high subtype (matrix-organizing)
    '2': 'NP_stress_response',       # AP-1/heat-shock stress state
    '3': 'NP_degenerative_UPR',      # UPR/autophagy degenerative state
    '4': 'AF_fibroblast',
    '5': 'NP_metallothionein',       # MT-high oxidative stress / senescent-like
    '6': 'T_NK_cell',
    '7': 'Pericyte_SMC',
    '8': 'Endothelial',
    '9': 'Erythrocyte',
    '10': 'Macrophage',
    '11': 'Monocyte_Neutrophil',
}

# Broader cell type grouping for compositional analysis
broad_annotation = {
    '0': 'NP_chondrocyte',
    '1': 'NP_chondrocyte',
    '2': 'NP_chondrocyte',
    '3': 'NP_chondrocyte',
    '4': 'AF_fibroblast',
    '5': 'NP_chondrocyte',
    '6': 'Immune',
    '7': 'Stromal_vascular',
    '8': 'Stromal_vascular',
    '9': 'Erythrocyte',
    '10': 'Immune',
    '11': 'Immune',
}

print("Manual annotation mapping:")
for k, v in cluster_annotation.items():
    print(f"  Cluster {k}: {v}  [{broad_annotation[k]}]")


Manual annotation mapping:
  Cluster 0: NP_chondrocyte  [NP_chondrocyte]
  Cluster 1: NP_chondrocyte_HAPLN1  [NP_chondrocyte]
  Cluster 2: NP_stress_response  [NP_chondrocyte]
  Cluster 3: NP_degenerative_UPR  [NP_chondrocyte]
  Cluster 4: AF_fibroblast  [AF_fibroblast]
  Cluster 5: NP_metallothionein  [NP_chondrocyte]
  Cluster 6: T_NK_cell  [Immune]
  Cluster 7: Pericyte_SMC  [Stromal_vascular]
  Cluster 8: Endothelial  [Stromal_vascular]
  Cluster 9: Erythrocyte  [Erythrocyte]
  Cluster 10: Macrophage  [Immune]
  Cluster 11: Monocyte_Neutrophil  [Immune]


Load annotated AnnData and apply manual cell type labels

In [3]:

import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Loading annotated AnnData...")
adata = sc.read_h5ad('/workspace/data/processed/adata_annotated.h5ad')
print(f"  Loaded: {adata.shape}")

# Apply manual annotations
adata.obs['cell_type'] = adata.obs['leiden_0.5'].map(cluster_annotation)
adata.obs['cell_type_broad'] = adata.obs['leiden_0.5'].map(broad_annotation)

# Verify
print("\nCell type counts (fine):")
print(adata.obs['cell_type'].value_counts())
print("\nCell type counts (broad):")
print(adata.obs['cell_type_broad'].value_counts())


Loading annotated AnnData...
  Loaded: (173628, 25304)

Cell type counts (fine):
cell_type
NP_chondrocyte           47120
NP_degenerative_UPR      34810
NP_metallothionein       31502
NP_chondrocyte_HAPLN1    17040
AF_fibroblast            16448
NP_stress_response        9967
Pericyte_SMC              4068
Macrophage                3482
Endothelial               3382
Monocyte_Neutrophil       2511
Erythrocyte               1658
T_NK_cell                 1640
Name: count, dtype: int64

Cell type counts (broad):
cell_type_broad
NP_chondrocyte      140439
AF_fibroblast        16448
Immune                7633
Stromal_vascular      7450
Erythrocyte           1658
Name: count, dtype: int64


Generate publication-quality annotated UMAP with manual cell type labels

In [5]:

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

# Color palette - colorblind-friendly + biologically meaningful
cell_type_colors = {
    'NP_chondrocyte':           '#2166AC',   # deep blue - canonical NP
    'NP_chondrocyte_HAPLN1':    '#4393C3',   # medium blue - HAPLN1 subtype
    'NP_stress_response':       '#92C5DE',   # light blue - stress state
    'NP_degenerative_UPR':      '#D1E5F0',   # very light blue - degenerative
    'NP_metallothionein':       '#74ADD1',   # steel blue - MT-high/senescent
    'AF_fibroblast':            '#D6604D',   # red-orange - AF
    'Pericyte_SMC':             '#F4A582',   # salmon - pericyte
    'Endothelial':              '#FDAE61',   # orange - endothelial
    'Macrophage':               '#4DAC26',   # green - macrophage
    'Monocyte_Neutrophil':      '#B8E186',   # light green - monocyte
    'T_NK_cell':                '#762A83',   # purple - T/NK
    'Erythrocyte':              '#E7D4E8',   # light purple - erythrocyte
}

# Get UMAP coordinates
umap = adata.obsm['X_umap']
cell_types = adata.obs['cell_type'].values

fig, axes = plt.subplots(1, 2, figsize=(20, 9))
fig.patch.set_facecolor('white')

# ── Panel 1: Fine cell types ──────────────────────────────────────────────────
ax = axes[0]
ax.set_facecolor('#F8F8F8')

# Plot each cell type
for ct, color in cell_type_colors.items():
    mask = cell_types == ct
    n = mask.sum()
    if n > 0:
        ax.scatter(umap[mask, 0], umap[mask, 1],
                   c=color, s=0.3, alpha=0.6, rasterized=True, label=f'{ct} (n={n:,})')

ax.set_xlabel('UMAP1', fontsize=12)
ax.set_ylabel('UMAP2', fontsize=12)
ax.set_title('Cell type annotation\n(173,628 cells, 7 datasets)', fontsize=13, fontweight='bold')
ax.tick_params(labelsize=9)

# Legend
handles = [mpatches.Patch(color=color, label=f'{ct}\n(n={int((cell_types==ct).sum()):,})')
           for ct, color in cell_type_colors.items() if (cell_types==ct).sum() > 0]
ax.legend(handles=handles, loc='upper left', fontsize=7.5, framealpha=0.9,
          ncol=1, markerscale=2, handlelength=1.5)

# ── Panel 2: Condition overlay ────────────────────────────────────────────────
ax2 = axes[1]
ax2.set_facecolor('#F8F8F8')

condition_colors = {
    'healthy':          '#2166AC',
    'mild_degen':       '#FEE090',
    'moderate_degen':   '#F46D43',
    'severe_degen':     '#A50026',
}
conditions = adata.obs['condition_harmonized'].values
for cond, color in condition_colors.items():
    mask = conditions == cond
    n = mask.sum()
    if n > 0:
        ax2.scatter(umap[mask, 0], umap[mask, 1],
                    c=color, s=0.3, alpha=0.5, rasterized=True, label=f'{cond} (n={n:,})')

ax2.set_xlabel('UMAP1', fontsize=12)
ax2.set_ylabel('UMAP2', fontsize=12)
ax2.set_title('Degeneration condition\n(healthy → severe)', fontsize=13, fontweight='bold')
ax2.tick_params(labelsize=9)

handles2 = [mpatches.Patch(color=color, label=f'{cond} (n={int((conditions==cond).sum()):,})')
            for cond, color in condition_colors.items() if (conditions==cond).sum() > 0]
ax2.legend(handles=handles2, loc='upper left', fontsize=9, framealpha=0.9)

plt.tight_layout(pad=2)
plt.savefig('/mnt/results/05_annotation/umap_annotated_v2.png', dpi=150, bbox_inches='tight')
plt.savefig('/mnt/results/05_annotation/umap_annotated_v2.svg', bbox_inches='tight')
plt.close()
print("Saved umap_annotated_v2.png/svg")


Saved umap_annotated_v2.png/svg


Save updated annotations to AnnData and obs CSV

In [7]:

import scanpy as sc
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Apply annotations (already loaded above)
adata.obs['cell_type'] = adata.obs['leiden_0.5'].map(cluster_annotation)
adata.obs['cell_type_broad'] = adata.obs['leiden_0.5'].map(broad_annotation)

# Save updated obs metadata
obs_out = adata.obs[['dataset', 'sample_id', 'donor_id', 'tissue', 'condition_harmonized',
                       'pfirrmann', 'age', 'sex', 'age_group',
                       'leiden_0.5', 'cell_type', 'cell_type_broad',
                       'n_genes_by_counts', 'total_counts', 'pct_counts_mt']].copy()
obs_out.index.name = 'cell_barcode'
obs_out.to_csv('/workspace/models/obs_annotated_manual.csv')
print(f"Saved obs_annotated_manual.csv: {obs_out.shape}")

# Also save to results for user
obs_out.to_csv('/mnt/results/05_annotation/cell_metadata.csv')
print("Saved cell_metadata.csv to results")

# Summary table
summary = obs_out.groupby(['cell_type', 'condition_harmonized']).size().unstack(fill_value=0)
summary['total'] = summary.sum(axis=1)
summary = summary.sort_values('total', ascending=False)
print("\nCell type × condition matrix:")
print(summary.to_string())


Saved obs_annotated_manual.csv: (173628, 15)
Saved cell_metadata.csv to results

Cell type × condition matrix:
condition_harmonized   healthy  mild_degen  moderate_degen  severe_degen  total
cell_type                                                                      
NP_chondrocyte           30244        7157            5683          4036  47120
NP_degenerative_UPR      19008        3382            9224          3196  34810
NP_metallothionein       22774        1781            5163          1784  31502
NP_chondrocyte_HAPLN1     8851        1528            4031          2630  17040
AF_fibroblast             6506         782            3848          5312  16448
NP_stress_response        5647        1075            1948          1297   9967
Pericyte_SMC              3302         123             254           389   4068
Macrophage                 822        1136            1412           112   3482
Endothelial               2471         105             157           649   3382
Monocyte_

Compositional analysis: cell type proportions per donor × condition with statistical testing

In [9]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('/mnt/results/06_composition', exist_ok=True)

obs = pd.read_csv('/workspace/models/obs_annotated_manual.csv', index_col=0)
print(f"Loaded obs: {obs.shape}")

# ── 1. Per-donor proportions ──────────────────────────────────────────────────
# Use fine cell types but group NP states for cleaner composition
obs['cell_type_plot'] = obs['cell_type'].replace({
    'NP_chondrocyte':        'NP: canonical',
    'NP_chondrocyte_HAPLN1': 'NP: HAPLN1+',
    'NP_stress_response':    'NP: stress',
    'NP_degenerative_UPR':   'NP: degenerative',
    'NP_metallothionein':    'NP: MT-high',
    'AF_fibroblast':         'AF fibroblast',
    'Pericyte_SMC':          'Pericyte/SMC',
    'Endothelial':           'Endothelial',
    'Macrophage':            'Macrophage',
    'Monocyte_Neutrophil':   'Monocyte/Neutrophil',
    'T_NK_cell':             'T/NK cell',
    'Erythrocyte':           'Erythrocyte',
})

# Condition order
cond_order = ['healthy', 'mild_degen', 'moderate_degen', 'severe_degen']
cond_labels = {'healthy': 'Healthy', 'mild_degen': 'Mild', 
               'moderate_degen': 'Moderate', 'severe_degen': 'Severe'}

# Per-donor cell type proportions
donor_props = (obs.groupby(['donor_id', 'condition_harmonized', 'cell_type_plot'])
               .size()
               .reset_index(name='n_cells'))
donor_totals = donor_props.groupby('donor_id')['n_cells'].sum().rename('total')
donor_props = donor_props.join(donor_totals, on='donor_id')
donor_props['proportion'] = donor_props['n_cells'] / donor_props['total']

# Only keep donors with ≥100 cells
valid_donors = donor_totals[donor_totals >= 100].index
donor_props = donor_props[donor_props['donor_id'].isin(valid_donors)]
print(f"Valid donors (≥100 cells): {len(valid_donors)}")

# ── 2. Mean proportions per condition ─────────────────────────────────────────
mean_props = (donor_props.groupby(['condition_harmonized', 'cell_type_plot'])['proportion']
              .mean().unstack(fill_value=0))
mean_props = mean_props.reindex(cond_order)
print("\nMean proportions per condition:")
print((mean_props * 100).round(1).to_string())


Loaded obs: (173628, 15)
Valid donors (≥100 cells): 29

Mean proportions per condition:
cell_type_plot        AF fibroblast  Endothelial  Erythrocyte  Macrophage  Monocyte/Neutrophil  NP: HAPLN1+  NP: MT-high  NP: canonical  NP: degenerative  NP: stress  Pericyte/SMC  T/NK cell
condition_harmonized                                                                                                                                                                          
healthy                         6.7          1.8          0.4         0.7                  0.1          8.8         18.0           29.8              22.8         5.1           2.5        0.2
mild_degen                      3.8          0.5          4.1         4.6                 13.2          7.2          8.8           34.1              16.7         5.3           0.6        4.4
moderate_degen                 10.7          0.4          0.7         3.5                  0.1         10.4         12.9           20.3             

Generate stacked bar + boxplot compositional figures

In [11]:

# Color palette for cell types
ct_colors = {
    'NP: canonical':      '#2166AC',
    'NP: HAPLN1+':        '#4393C3',
    'NP: stress':         '#92C5DE',
    'NP: degenerative':   '#D1E5F0',
    'NP: MT-high':        '#74ADD1',
    'AF fibroblast':      '#D6604D',
    'Pericyte/SMC':       '#F4A582',
    'Endothelial':        '#FDAE61',
    'Macrophage':         '#4DAC26',
    'Monocyte/Neutrophil':'#B8E186',
    'T/NK cell':          '#762A83',
    'Erythrocyte':        '#E7D4E8',
}

ct_order = list(ct_colors.keys())

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
fig.patch.set_facecolor('white')

# ── Panel A: Stacked bar (mean proportions per condition) ─────────────────────
ax = axes[0]
bottom = np.zeros(4)
x = np.arange(4)
for ct in ct_order:
    if ct in mean_props.columns:
        vals = mean_props[ct].values
        ax.bar(x, vals, bottom=bottom, color=ct_colors[ct], label=ct, width=0.65, edgecolor='white', linewidth=0.3)
        bottom += vals

ax.set_xticks(x)
ax.set_xticklabels(['Healthy', 'Mild', 'Moderate', 'Severe'], fontsize=12)
ax.set_ylabel('Mean cell type proportion', fontsize=12)
ax.set_title('Cell type composition\nby degeneration grade', fontsize=13, fontweight='bold')
ax.set_ylim(0, 1.02)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Legend outside
handles = [mpatches.Patch(color=ct_colors[ct], label=ct) for ct in ct_order if ct in mean_props.columns]
ax.legend(handles=handles, bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8.5,
          framealpha=0.9, title='Cell type', title_fontsize=9)

# ── Panel B: Boxplot of key cell types across conditions ──────────────────────
ax2 = axes[1]

# Focus on the most biologically interesting changes
focus_cts = ['NP: canonical', 'NP: degenerative', 'NP: MT-high', 'AF fibroblast', 'Macrophage']
focus_colors = [ct_colors[ct] for ct in focus_cts]

# Pivot for plotting
plot_data = donor_props[donor_props['cell_type_plot'].isin(focus_cts)].copy()
plot_data['condition_label'] = plot_data['condition_harmonized'].map(cond_labels)
plot_data['condition_label'] = pd.Categorical(plot_data['condition_label'],
                                               categories=['Healthy', 'Mild', 'Moderate', 'Severe'])

# Jitter + box per cell type
n_ct = len(focus_cts)
x_positions = np.arange(n_ct)
width = 0.18
offsets = np.linspace(-0.27, 0.27, 4)
cond_plot_colors = {'Healthy': '#2166AC', 'Mild': '#FEE090', 'Moderate': '#F46D43', 'Severe': '#A50026'}

for ci, (cond, clabel) in enumerate(cond_labels.items()):
    for xi, ct in enumerate(focus_cts):
        sub = plot_data[(plot_data['condition_harmonized'] == cond) &
                        (plot_data['cell_type_plot'] == ct)]['proportion'].values
        if len(sub) == 0:
            continue
        xpos = xi + offsets[ci]
        bp = ax2.boxplot(sub, positions=[xpos], widths=0.13,
                         patch_artist=True, showfliers=False,
                         boxprops=dict(facecolor=cond_plot_colors[clabel], alpha=0.7),
                         medianprops=dict(color='black', linewidth=1.5),
                         whiskerprops=dict(color='gray', linewidth=0.8),
                         capprops=dict(color='gray', linewidth=0.8))
        # Jitter
        jitter = np.random.uniform(-0.04, 0.04, len(sub))
        ax2.scatter(xpos + jitter, sub, color=cond_plot_colors[clabel],
                    s=18, alpha=0.8, zorder=3, edgecolors='white', linewidths=0.3)

ax2.set_xticks(x_positions)
ax2.set_xticklabels([ct.replace(' ', '\n') for ct in focus_cts], fontsize=9)
ax2.set_ylabel('Proportion of cells', fontsize=12)
ax2.set_title('Key cell type proportions\nper donor by condition', fontsize=13, fontweight='bold')
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda y, _: f'{y:.0%}'))
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)

handles2 = [mpatches.Patch(color=c, label=l) for l, c in cond_plot_colors.items()]
ax2.legend(handles=handles2, title='Condition', fontsize=9, title_fontsize=9,
           loc='upper right', framealpha=0.9)

plt.tight_layout(pad=2)
plt.savefig('/mnt/results/06_composition/composition_overview.png', dpi=150, bbox_inches='tight')
plt.savefig('/mnt/results/06_composition/composition_overview.svg', bbox_inches='tight')
plt.close()
print("Saved composition_overview.png/svg")


Saved composition_overview.png/svg


Verify compositional data integrity - cross-check against raw cell counts

In [13]:

import pandas as pd
import numpy as np

# Reload from disk to confirm data source
obs_check = pd.read_csv('/workspace/models/obs_annotated_manual.csv', index_col=0)
print(f"File: /workspace/models/obs_annotated_manual.csv")
print(f"Shape: {obs_check.shape}")
print(f"Columns: {list(obs_check.columns)}")
print(f"\nFirst 3 rows:")
print(obs_check.head(3).to_string())

# Cross-check total cell counts match what was computed during annotation
print(f"\n--- Cross-check: total cells per condition ---")
cond_counts = obs_check['condition_harmonized'].value_counts()
print(cond_counts)
expected = {'healthy': 100234, 'mild_degen': 21646, 'moderate_degen': 32138, 'severe_degen': 19610}
for k, v in expected.items():
    actual = cond_counts.get(k, 0)
    match = "✓" if actual == v else f"✗ (expected {v})"
    print(f"  {k}: {actual} {match}")

# Cross-check cell type totals
print(f"\n--- Cross-check: cell type totals ---")
ct_counts = obs_check['cell_type'].value_counts()
expected_ct = {
    'NP_chondrocyte': 47120, 'NP_degenerative_UPR': 34810, 'NP_metallothionein': 31502,
    'NP_chondrocyte_HAPLN1': 17040, 'AF_fibroblast': 16448, 'NP_stress_response': 9967,
    'Pericyte_SMC': 4068, 'Macrophage': 3482, 'Endothelial': 3382,
    'Monocyte_Neutrophil': 2511, 'Erythrocyte': 1658, 'T_NK_cell': 1640
}
for k, v in expected_ct.items():
    actual = ct_counts.get(k, 0)
    match = "✓" if actual == v else f"✗ (expected {v})"
    print(f"  {k}: {actual} {match}")

# Verify donor proportions are real (not random)
print(f"\n--- Donor count check ---")
print(f"Unique donors: {obs_check['donor_id'].nunique()}")
print(f"Unique datasets: {obs_check['dataset'].nunique()}")
print(f"Unique samples: {obs_check['sample_id'].nunique()}")

# Spot-check: donor with most cells
top_donor = obs_check['donor_id'].value_counts().index[0]
top_n = obs_check['donor_id'].value_counts().iloc[0]
top_ct = obs_check[obs_check['donor_id'] == top_donor]['cell_type'].value_counts()
print(f"\nTop donor '{top_donor}' ({top_n} cells):")
print(top_ct.head(5))


File: /workspace/models/obs_annotated_manual.csv
Shape: (173628, 15)
Columns: ['dataset', 'sample_id', 'donor_id', 'tissue', 'condition_harmonized', 'pfirrmann', 'age', 'sex', 'age_group', 'leiden_0.5', 'cell_type', 'cell_type_broad', 'n_genes_by_counts', 'total_counts', 'pct_counts_mt']

First 3 rows:
                                         dataset   sample_id donor_id tissue condition_harmonized  pfirrmann  age  sex age_group  leiden_0.5            cell_type   cell_type_broad  n_genes_by_counts  total_counts  pct_counts_mt
cell_barcode                                                                                                                                                                                                                       
GSM4878538_hNP_1_AAACCCACAAAGACTA-1-1  GSE160756  GSM4878538      NP1     NP              healthy          1  NaN  NaN   unknown           0       NP_chondrocyte    NP_chondrocyte               1356        3157.0       1.203674
GSM4878538_h

Statistical testing of compositional changes across degeneration grades (Kruskal-Wallis + pairwise)

In [15]:

import pandas as pd
import numpy as np
from scipy import stats
from itertools import combinations
import warnings
warnings.filterwarnings('ignore')

obs = pd.read_csv('/workspace/models/obs_annotated_manual.csv', index_col=0)

# Map to plot labels
obs['cell_type_plot'] = obs['cell_type'].replace({
    'NP_chondrocyte':        'NP: canonical',
    'NP_chondrocyte_HAPLN1': 'NP: HAPLN1+',
    'NP_stress_response':    'NP: stress',
    'NP_degenerative_UPR':   'NP: degenerative',
    'NP_metallothionein':    'NP: MT-high',
    'AF_fibroblast':         'AF fibroblast',
    'Pericyte_SMC':          'Pericyte/SMC',
    'Endothelial':           'Endothelial',
    'Macrophage':            'Macrophage',
    'Monocyte_Neutrophil':   'Monocyte/Neutrophil',
    'T_NK_cell':             'T/NK cell',
    'Erythrocyte':           'Erythrocyte',
})

# Per-donor proportions
donor_props = (obs.groupby(['donor_id', 'condition_harmonized', 'cell_type_plot'])
               .size().reset_index(name='n_cells'))
donor_totals = donor_props.groupby('donor_id')['n_cells'].sum().rename('total')
donor_props = donor_props.join(donor_totals, on='donor_id')
donor_props['proportion'] = donor_props['n_cells'] / donor_props['total']

# Pivot to wide format: donors × cell types
prop_wide = donor_props.pivot_table(index=['donor_id', 'condition_harmonized'],
                                     columns='cell_type_plot', values='proportion',
                                     fill_value=0).reset_index()

cond_order = ['healthy', 'mild_degen', 'moderate_degen', 'severe_degen']
cell_types = [c for c in prop_wide.columns if c not in ['donor_id', 'condition_harmonized']]

# Kruskal-Wallis test per cell type
print("Kruskal-Wallis test (H statistic, p-value) across 4 conditions:")
print(f"{'Cell type':<28} {'H':>8} {'p-value':>12} {'sig':>6}")
print("-" * 58)

kw_results = []
for ct in cell_types:
    groups = [prop_wide[prop_wide['condition_harmonized'] == c][ct].values
              for c in cond_order if c in prop_wide['condition_harmonized'].values]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        H, p = stats.kruskal(*groups)
        sig = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
        print(f"{ct:<28} {H:>8.2f} {p:>12.4f} {sig:>6}")
        kw_results.append({'cell_type': ct, 'H': H, 'p_kruskal': p, 'sig': sig})

kw_df = pd.DataFrame(kw_results).sort_values('p_kruskal')
kw_df.to_csv('/mnt/results/06_composition/kruskal_wallis_results.csv', index=False)
print("\nSaved kruskal_wallis_results.csv")


Kruskal-Wallis test (H statistic, p-value) across 4 conditions:
Cell type                           H      p-value    sig
----------------------------------------------------------
AF fibroblast                    8.45       0.0376      *
Endothelial                      0.72       0.8673     ns
Erythrocyte                      0.73       0.8653     ns
Macrophage                       4.00       0.2610     ns
Monocyte/Neutrophil              5.46       0.1412     ns
NP: HAPLN1+                      2.19       0.5334     ns
NP: MT-high                      3.37       0.3382     ns
NP: canonical                    5.01       0.1712     ns
NP: degenerative                 2.38       0.4976     ns
NP: stress                       1.78       0.6195     ns
Pericyte/SMC                     2.74       0.4338     ns
T/NK cell                        3.81       0.2828     ns

Saved kruskal_wallis_results.csv


Build pseudobulk count matrices per donor × cell type for DESeq2

In [17]:

import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp
import os
import warnings
warnings.filterwarnings('ignore')

os.makedirs('/workspace/data/pseudobulk', exist_ok=True)
os.makedirs('/mnt/results/07_pseudobulk', exist_ok=True)

print("Loading integrated AnnData (raw counts needed for pseudobulk)...")
adata = sc.read_h5ad('/workspace/data/processed/adata_annotated.h5ad')
print(f"  Loaded: {adata.shape}")

# Apply manual annotations
cluster_annotation = {
    '0': 'NP_chondrocyte', '1': 'NP_chondrocyte_HAPLN1', '2': 'NP_stress_response',
    '3': 'NP_degenerative_UPR', '4': 'AF_fibroblast', '5': 'NP_metallothionein',
    '6': 'T_NK_cell', '7': 'Pericyte_SMC', '8': 'Endothelial',
    '9': 'Erythrocyte', '10': 'Macrophage', '11': 'Monocyte_Neutrophil',
}
adata.obs['cell_type'] = adata.obs['leiden_0.5'].map(cluster_annotation)

# Check if raw counts are available
print(f"\nadata.X dtype: {adata.X.dtype}")
print(f"adata.X min/max: {adata.X.min():.2f} / {adata.X.max():.2f}")
print(f"adata.layers keys: {list(adata.layers.keys())}")
print(f"adata.raw: {adata.raw}")

# The X matrix at this point is log-normalized (from integration pipeline)
# We need to back-calculate raw counts from adata.raw if available,
# or use the raw concat file


Loading integrated AnnData (raw counts needed for pseudobulk)...
  Loaded: (173628, 25304)

adata.X dtype: float64
adata.X min/max: 0.00 / 8.97
adata.layers keys: ['counts', 'lognorm']
adata.raw: None


Build pseudobulk matrices from raw counts layer, aggregate per donor × cell type

In [19]:

import pandas as pd
import numpy as np
import scipy.sparse as sp
import os

# Raw counts are in adata.layers['counts'] — perfect
print("Using adata.layers['counts'] for pseudobulk aggregation")
print(f"  counts layer dtype: {adata.layers['counts'].dtype}")
print(f"  counts min/max: {adata.layers['counts'].min():.1f} / {adata.layers['counts'].max():.1f}")

# Focus on cell types with enough cells for pseudobulk:
# Need ≥3 donors per condition and ≥10 cells per donor×cell_type
focus_cell_types = ['NP_chondrocyte', 'NP_chondrocyte_HAPLN1', 'NP_stress_response',
                    'NP_degenerative_UPR', 'NP_metallothionein', 'AF_fibroblast']

genes = adata.var_names.tolist()
print(f"\nGenes: {len(genes)}")

# Build pseudobulk per cell type
pb_summary = []
for ct in focus_cell_types:
    ct_mask = adata.obs['cell_type'] == ct
    ct_adata = adata[ct_mask]
    
    # Group by donor × condition
    groups = ct_adata.obs.groupby(['donor_id', 'condition_harmonized']).size()
    groups = groups[groups >= 10]  # min 10 cells per pseudobulk sample
    
    print(f"\n{ct}: {ct_mask.sum()} cells, {len(groups)} valid donor×condition groups")
    
    # Aggregate counts
    pb_counts = {}
    pb_meta = {}
    for (donor, cond), n_cells in groups.items():
        key = f"{donor}__{cond}"
        mask = (ct_adata.obs['donor_id'] == donor) & (ct_adata.obs['condition_harmonized'] == cond)
        X_sub = ct_adata.layers['counts'][mask]
        if sp.issparse(X_sub):
            agg = np.array(X_sub.sum(axis=0)).flatten()
        else:
            agg = X_sub.sum(axis=0)
        pb_counts[key] = agg
        pb_meta[key] = {'donor_id': donor, 'condition': cond, 'n_cells': n_cells}
    
    if len(pb_counts) < 4:
        print(f"  Skipping {ct}: too few pseudobulk samples ({len(pb_counts)})")
        continue
    
    # Save count matrix and metadata
    pb_df = pd.DataFrame(pb_counts, index=genes).astype(int)
    meta_df = pd.DataFrame(pb_meta).T
    
    ct_safe = ct.replace('/', '_')
    pb_df.to_csv(f'/workspace/data/pseudobulk/{ct_safe}_counts.csv')
    meta_df.to_csv(f'/workspace/data/pseudobulk/{ct_safe}_meta.csv')
    
    pb_summary.append({
        'cell_type': ct,
        'n_cells': ct_mask.sum(),
        'n_pb_samples': len(pb_counts),
        'conditions': meta_df['condition'].value_counts().to_dict()
    })
    print(f"  Saved: {pb_df.shape[1]} samples × {pb_df.shape[0]} genes")
    print(f"  Conditions: {meta_df['condition'].value_counts().to_dict()}")

print("\n=== Pseudobulk summary ===")
for s in pb_summary:
    print(f"  {s['cell_type']}: {s['n_pb_samples']} samples, {s['conditions']}")


Using adata.layers['counts'] for pseudobulk aggregation
  counts layer dtype: float64
  counts min/max: 0.0 / 14475.0

Genes: 25304

NP_chondrocyte: 47120 cells, 30 valid donor×condition groups
Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 38, in <module>
  File "/opt/conda/lib/python3.11/site-packages/scipy/sparse/_index.py", line 30, in __getitem__
    index, new_shape = self._validate_indices(key)
                       ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/scipy/sparse/_index.py", line 269, in _validate_indices
    index.extend(ix.nonzero())
                 ^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/pandas/core/generic.py", line 6318, in __getattr__
    return object.__getattribute__(self, name)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'Series' object has no attribute 'nonzero'


Fix boolean indexing for sparse matrix slicing in pseudobulk aggregation

In [21]:

import pandas as pd
import numpy as np
import scipy.sparse as sp
import os

focus_cell_types = ['NP_chondrocyte', 'NP_chondrocyte_HAPLN1', 'NP_stress_response',
                    'NP_degenerative_UPR', 'NP_metallothionein', 'AF_fibroblast']

genes = adata.var_names.tolist()
pb_summary = []

for ct in focus_cell_types:
    ct_idx = np.where(adata.obs['cell_type'].values == ct)[0]
    ct_adata = adata[ct_idx]
    
    # Group by donor × condition
    groups = ct_adata.obs.groupby(['donor_id', 'condition_harmonized']).size()
    groups = groups[groups >= 10]
    
    print(f"\n{ct}: {len(ct_idx)} cells, {len(groups)} valid donor×condition groups")
    
    pb_counts = {}
    pb_meta = {}
    for (donor, cond), n_cells in groups.items():
        key = f"{donor}__{cond}"
        sub_idx = np.where(
            (ct_adata.obs['donor_id'].values == donor) &
            (ct_adata.obs['condition_harmonized'].values == cond)
        )[0]
        X_sub = ct_adata.layers['counts'][sub_idx]
        if sp.issparse(X_sub):
            agg = np.array(X_sub.sum(axis=0)).flatten()
        else:
            agg = X_sub.sum(axis=0).flatten()
        pb_counts[key] = agg
        pb_meta[key] = {'donor_id': donor, 'condition': cond, 'n_cells': int(n_cells)}
    
    if len(pb_counts) < 4:
        print(f"  Skipping: too few pseudobulk samples ({len(pb_counts)})")
        continue
    
    pb_df = pd.DataFrame(pb_counts, index=genes).astype(int)
    meta_df = pd.DataFrame(pb_meta).T
    
    ct_safe = ct.replace('/', '_')
    pb_df.to_csv(f'/workspace/data/pseudobulk/{ct_safe}_counts.csv')
    meta_df.to_csv(f'/workspace/data/pseudobulk/{ct_safe}_meta.csv')
    
    pb_summary.append({
        'cell_type': ct, 'n_cells': len(ct_idx),
        'n_pb_samples': len(pb_counts),
        'conditions': meta_df['condition'].value_counts().to_dict()
    })
    print(f"  Saved: {pb_df.shape[1]} samples × {pb_df.shape[0]} genes")
    print(f"  Conditions: {meta_df['condition'].value_counts().to_dict()}")

print("\n=== Pseudobulk aggregation complete ===")
for s in pb_summary:
    print(f"  {s['cell_type']}: {s['n_pb_samples']} samples | {s['conditions']}")



NP_chondrocyte: 47120 cells, 30 valid donor×condition groups
  Saved: 30 samples × 25304 genes
  Conditions: {'healthy': 15, 'moderate_degen': 8, 'mild_degen': 4, 'severe_degen': 3}

NP_chondrocyte_HAPLN1: 17040 cells, 30 valid donor×condition groups
  Saved: 30 samples × 25304 genes
  Conditions: {'healthy': 15, 'moderate_degen': 8, 'mild_degen': 4, 'severe_degen': 3}

NP_stress_response: 9967 cells, 30 valid donor×condition groups
  Saved: 30 samples × 25304 genes
  Conditions: {'healthy': 15, 'moderate_degen': 8, 'mild_degen': 4, 'severe_degen': 3}

NP_degenerative_UPR: 34810 cells, 30 valid donor×condition groups
  Saved: 30 samples × 25304 genes
  Conditions: {'healthy': 15, 'moderate_degen': 8, 'mild_degen': 4, 'severe_degen': 3}

NP_metallothionein: 31502 cells, 30 valid donor×condition groups
  Saved: 30 samples × 25304 genes
  Conditions: {'healthy': 15, 'moderate_degen': 8, 'mild_degen': 4, 'severe_degen': 3}

AF_fibroblast: 16448 cells, 30 valid donor×condition groups
  Sav

DESeq2 pseudobulk DE: severe_degen vs healthy for all 6 cell types

In [23]:

suppressPackageStartupMessages({
  library(DESeq2)
  library(dplyr)
  library(readr)
})

cell_types <- c("NP_chondrocyte", "NP_chondrocyte_HAPLN1", "NP_stress_response",
                "NP_degenerative_UPR", "NP_metallothionein", "AF_fibroblast")

contrast_pairs <- list(
  c("severe_degen", "healthy"),
  c("moderate_degen", "healthy"),
  c("mild_degen", "healthy")
)

all_results <- list()

for (ct in cell_types) {
  cat(sprintf("\n=== %s ===\n", ct))
  
  counts_file <- sprintf("/workspace/data/pseudobulk/%s_counts.csv", ct)
  meta_file   <- sprintf("/workspace/data/pseudobulk/%s_meta.csv", ct)
  
  counts <- read.csv(counts_file, row.names=1, check.names=FALSE)
  meta   <- read.csv(meta_file, row.names=1)
  
  # Align
  meta <- meta[colnames(counts), , drop=FALSE]
  meta$condition <- factor(meta$condition,
                           levels=c("healthy","mild_degen","moderate_degen","severe_degen"))
  
  cat(sprintf("  Counts: %d genes × %d samples\n", nrow(counts), ncol(counts)))
  cat(sprintf("  Conditions: %s\n", paste(table(meta$condition), collapse=" / ")))
  
  # Filter low-count genes: keep genes with ≥10 counts in ≥3 samples
  keep <- rowSums(counts >= 10) >= 3
  counts_filt <- counts[keep, ]
  cat(sprintf("  Genes after filtering: %d\n", nrow(counts_filt)))
  
  # Build DESeq2 object
  dds <- DESeqDataSetFromMatrix(
    countData = as.matrix(counts_filt),
    colData   = meta,
    design    = ~ condition
  )
  
  # Run DESeq2
  dds <- tryCatch(
    DESeq(dds, quiet=TRUE),
    error = function(e) { cat("  DESeq2 error:", conditionMessage(e), "\n"); NULL }
  )
  if (is.null(dds)) next
  
  # Extract results for each contrast
  for (pair in contrast_pairs) {
    test_cond <- pair[1]; ref_cond <- pair[2]
    
    # Check both conditions have samples
    if (!(test_cond %in% levels(meta$condition)) ||
        sum(meta$condition == test_cond) == 0) next
    
    res <- tryCatch(
      results(dds, contrast=c("condition", test_cond, ref_cond),
              alpha=0.05, independentFiltering=TRUE),
      error = function(e) NULL
    )
    if (is.null(res)) next
    
    res_df <- as.data.frame(res) %>%
      tibble::rownames_to_column("gene") %>%
      filter(!is.na(padj)) %>%
      arrange(padj)
    
    n_sig <- sum(res_df$padj < 0.05, na.rm=TRUE)
    n_up  <- sum(res_df$padj < 0.05 & res_df$log2FoldChange > 0, na.rm=TRUE)
    n_dn  <- sum(res_df$padj < 0.05 & res_df$log2FoldChange < 0, na.rm=TRUE)
    cat(sprintf("  %s vs healthy: %d DEGs (%d up, %d down)\n",
                test_cond, n_sig, n_up, n_dn))
    
    key <- sprintf("%s__%s_vs_healthy", ct, test_cond)
    all_results[[key]] <- res_df
    
    # Save
    out_file <- sprintf("/workspace/data/pseudobulk/deseq2_%s__%s_vs_healthy.csv", ct, test_cond)
    write_csv(res_df, out_file)
  }
}

cat("\n=== DESeq2 complete ===\n")
cat(sprintf("Total contrasts computed: %d\n", length(all_results)))



=== NP_chondrocyte ===
  Counts: 25304 genes <c3><97> 30 samples
  Conditions: 15 / 4 / 8 / 3
  Genes after filtering: 16948
  severe_degen vs healthy: 2641 DEGs (332 up, 2309 down)
  moderate_degen vs healthy: 1018 DEGs (521 up, 497 down)
  mild_degen vs healthy: 2792 DEGs (230 up, 2562 down)

=== NP_chondrocyte_HAPLN1 ===
  Counts: 25304 genes <c3><97> 30 samples
  Conditions: 15 / 4 / 8 / 3
  Genes after filtering: 14936
  severe_degen vs healthy: 1786 DEGs (223 up, 1563 down)
  moderate_degen vs healthy: 689 DEGs (288 up, 401 down)
  mild_degen vs healthy: 1485 DEGs (165 up, 1320 down)

=== NP_stress_response ===
  Counts: 25304 genes <c3><97> 30 samples
  Conditions: 15 / 4 / 8 / 3
  Genes after filtering: 12866
  severe_degen vs healthy: 1081 DEGs (104 up, 977 down)
  moderate_degen vs healthy: 216 DEGs (68 up, 148 down)
  mild_degen vs healthy: 1082 DEGs (149 up, 933 down)

=== NP_degenerative_UPR ===
  Counts: 25304 genes <c3><97> 30 samples
  Conditions: 15 / 4 / 8 / 3
  Gene

Generate volcano plots for severe_degen vs healthy across all 6 cell types

In [25]:

suppressPackageStartupMessages({
  library(ggplot2)
  library(dplyr)
  library(readr)
  library(ggrepel)
  library(patchwork)
})

cell_types <- c("NP_chondrocyte", "NP_chondrocyte_HAPLN1", "NP_stress_response",
                "NP_degenerative_UPR", "NP_metallothionein", "AF_fibroblast")

ct_labels <- c(
  "NP_chondrocyte"        = "NP: canonical",
  "NP_chondrocyte_HAPLN1" = "NP: HAPLN1+",
  "NP_stress_response"    = "NP: stress",
  "NP_degenerative_UPR"   = "NP: degenerative",
  "NP_metallothionein"    = "NP: MT-high",
  "AF_fibroblast"         = "AF fibroblast"
)

# Genes to label on volcano plots (IVD-relevant)
label_genes <- c("ACAN","COL2A1","COL1A1","COL1A2","MMP3","MMP13","ADAMTS5",
                 "IL6","IL1B","TNF","CXCL8","CCL2","FN1","VIM","SPARC",
                 "MT1G","MT2A","SOD2","HMOX1","HSPA1A","JUN","FOS",
                 "SQSTM1","ATF3","DDIT3","VEGFA","HIF1A","CDKN1A","TP53",
                 "HAPLN1","COMP","CILP","LOXL2","POSTN","CTGF","SPP1",
                 "KRT18","KRT19","TBXT","SOX9","RUNX2","COL10A1")

plots <- list()

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  
  res <- read_csv(f, show_col_types=FALSE) %>%
    filter(!is.na(padj), !is.na(log2FoldChange)) %>%
    mutate(
      neg_log10_padj = pmin(-log10(padj), 50),
      sig = case_when(
        padj < 0.05 & log2FoldChange >  1 ~ "Up",
        padj < 0.05 & log2FoldChange < -1 ~ "Down",
        padj < 0.05                        ~ "Sig (|LFC|<1)",
        TRUE                               ~ "NS"
      ),
      label = ifelse(gene %in% label_genes & padj < 0.05, gene, NA)
    )
  
  n_up   <- sum(res$sig == "Up",   na.rm=TRUE)
  n_dn   <- sum(res$sig == "Down", na.rm=TRUE)
  n_sig  <- sum(res$padj < 0.05,  na.rm=TRUE)
  
  p <- ggplot(res, aes(x=log2FoldChange, y=neg_log10_padj, color=sig)) +
    geom_point(size=0.4, alpha=0.5) +
    geom_text_repel(aes(label=label), size=2.2, max.overlaps=20,
                    segment.size=0.2, segment.alpha=0.5,
                    box.padding=0.3, force=2, na.rm=TRUE) +
    geom_vline(xintercept=c(-1,1), linetype="dashed", color="gray50", linewidth=0.4) +
    geom_hline(yintercept=-log10(0.05), linetype="dashed", color="gray50", linewidth=0.4) +
    scale_color_manual(values=c("Up"="#D6604D","Down"="#4393C3",
                                "Sig (|LFC|<1)"="#AAAAAA","NS"="#DDDDDD")) +
    labs(title=ct_labels[ct],
         subtitle=sprintf("%d DEGs: %d↑  %d↓", n_sig, n_up, n_dn),
         x="log2 fold change (severe vs healthy)",
         y="-log10(padj)") +
    theme_classic(base_size=9) +
    theme(legend.position="none",
          plot.title=element_text(face="bold", size=9),
          plot.subtitle=element_text(size=7.5, color="gray30")) +
    xlim(c(-8, 8))
  
  plots[[ct]] <- p
}

# Combine into 2×3 grid
combined <- (plots[[1]] | plots[[2]] | plots[[3]]) /
            (plots[[4]] | plots[[5]] | plots[[6]]) +
  plot_annotation(
    title="Pseudobulk DESeq2: severe degeneration vs healthy",
    subtitle="173,628 cells | 29 donors | padj < 0.05, |LFC| > 1 highlighted",
    theme=theme(plot.title=element_text(face="bold", size=13),
                plot.subtitle=element_text(size=9, color="gray30"))
  )

ggsave("/mnt/results/07_pseudobulk/volcano_severe_vs_healthy.png",
       combined, width=15, height=10, dpi=150)
ggsave("/mnt/results/07_pseudobulk/volcano_severe_vs_healthy.svg",
       combined, width=15, height=10)
cat("Saved volcano_severe_vs_healthy.png/svg\n")


Error: The package "svglite" is required to save as SVG. Warning: package 'ggplot2' was built under R version 4.4.3

Compile and save top DEG tables for all contrasts, then run pathway enrichment with clusterProfiler

In [27]:

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(clusterProfiler)
  library(org.Hs.eg.db)
  library(enrichplot)
  library(ggplot2)
  library(patchwork)
})

dir.create("/mnt/results/07_pseudobulk", showWarnings=FALSE)
dir.create("/mnt/results/08_pathways", showWarnings=FALSE)

cell_types <- c("NP_chondrocyte", "NP_chondrocyte_HAPLN1", "NP_stress_response",
                "NP_degenerative_UPR", "NP_metallothionein", "AF_fibroblast")

ct_labels <- c(
  "NP_chondrocyte"        = "NP: canonical",
  "NP_chondrocyte_HAPLN1" = "NP: HAPLN1+",
  "NP_stress_response"    = "NP: stress",
  "NP_degenerative_UPR"   = "NP: degenerative",
  "NP_metallothionein"    = "NP: MT-high",
  "AF_fibroblast"         = "AF fibroblast"
)

# ── 1. Compile master DEG table (severe vs healthy) ───────────────────────────
all_degs <- list()
for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  df <- read_csv(f, show_col_types=FALSE) %>%
    filter(!is.na(padj)) %>%
    mutate(cell_type=ct, contrast="severe_vs_healthy")
  all_degs[[ct]] <- df
}
master_degs <- bind_rows(all_degs)
write_csv(master_degs, "/mnt/results/07_pseudobulk/all_DEGs_severe_vs_healthy.csv")
cat(sprintf("Master DEG table: %d rows saved\n", nrow(master_degs)))

# Top 20 per cell type
top20 <- master_degs %>%
  filter(padj < 0.05) %>%
  group_by(cell_type) %>%
  slice_min(padj, n=20) %>%
  select(cell_type, gene, log2FoldChange, padj, baseMean)
write_csv(top20, "/mnt/results/07_pseudobulk/top20_DEGs_per_celltype.csv")
cat(sprintf("Top-20 DEGs table: %d rows\n", nrow(top20)))

# ── 2. ORA pathway enrichment (KEGG) for each cell type ──────────────────────
cat("\n=== ORA pathway enrichment (KEGG) ===\n")
ora_results <- list()

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  
  res <- read_csv(f, show_col_types=FALSE) %>% filter(!is.na(padj))
  
  # Significant DEGs (padj < 0.05, |LFC| > 0.5)
  sig_genes <- res %>% filter(padj < 0.05, abs(log2FoldChange) > 0.5) %>% pull(gene)
  bg_genes  <- res %>% pull(gene)
  
  # Convert to Entrez IDs
  sig_entrez <- bitr(sig_genes, fromType="SYMBOL", toType="ENTREZID",
                     OrgDb=org.Hs.eg.db, drop=TRUE)$ENTREZID
  bg_entrez  <- bitr(bg_genes,  fromType="SYMBOL", toType="ENTREZID",
                     OrgDb=org.Hs.eg.db, drop=TRUE)$ENTREZID
  
  if (length(sig_entrez) < 10) {
    cat(sprintf("  %s: too few genes (%d), skipping\n", ct, length(sig_entrez)))
    next
  }
  
  # KEGG ORA
  kegg <- tryCatch(
    enrichKEGG(gene=sig_entrez, universe=bg_entrez, organism="hsa",
               pAdjustMethod="BH", pvalueCutoff=0.05, qvalueCutoff=0.2),
    error=function(e) NULL
  )
  
  if (!is.null(kegg) && nrow(kegg@result) > 0) {
    n_sig <- sum(kegg@result$p.adjust < 0.05)
    cat(sprintf("  %s: %d sig KEGG pathways\n", ct_labels[ct], n_sig))
    ora_results[[ct]] <- kegg
  } else {
    cat(sprintf("  %s: no significant KEGG pathways\n", ct_labels[ct]))
  }
}

cat(sprintf("\nORA complete: %d cell types with results\n", length(ora_results)))


Master DEG table: 89457 rows saved
Error: unable to find an inherited method for function 'select' for signature 'x = "grouped_df"' 

Fix dplyr/AnnotationDbi conflict, save top DEG table, run KEGG ORA enrichment

In [29]:

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(clusterProfiler)
  library(org.Hs.eg.db)
  library(ggplot2)
  library(patchwork)
})

# Explicitly resolve the conflict
select <- dplyr::select

dir.create("/mnt/results/07_pseudobulk", showWarnings=FALSE)
dir.create("/mnt/results/08_pathways", showWarnings=FALSE)

cell_types <- c("NP_chondrocyte", "NP_chondrocyte_HAPLN1", "NP_stress_response",
                "NP_degenerative_UPR", "NP_metallothionein", "AF_fibroblast")
ct_labels <- c(
  "NP_chondrocyte"        = "NP: canonical",
  "NP_chondrocyte_HAPLN1" = "NP: HAPLN1+",
  "NP_stress_response"    = "NP: stress",
  "NP_degenerative_UPR"   = "NP: degenerative",
  "NP_metallothionein"    = "NP: MT-high",
  "AF_fibroblast"         = "AF fibroblast"
)

# ── Top-20 DEG table ──────────────────────────────────────────────────────────
all_degs <- lapply(cell_types, function(ct) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) return(NULL)
  read_csv(f, show_col_types=FALSE) %>%
    filter(!is.na(padj)) %>%
    mutate(cell_type=ct, contrast="severe_vs_healthy")
})
master_degs <- bind_rows(all_degs)

top20 <- master_degs %>%
  filter(padj < 0.05) %>%
  group_by(cell_type) %>%
  slice_min(order_by=padj, n=20) %>%
  dplyr::select(cell_type, gene, log2FoldChange, padj, baseMean) %>%
  ungroup()
write_csv(top20, "/mnt/results/07_pseudobulk/top20_DEGs_per_celltype.csv")
cat(sprintf("Top-20 DEGs table saved: %d rows\n", nrow(top20)))

# ── KEGG ORA per cell type ────────────────────────────────────────────────────
cat("\n=== KEGG ORA (severe vs healthy, padj<0.05, |LFC|>0.5) ===\n")
ora_results <- list()

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  res <- read_csv(f, show_col_types=FALSE) %>% filter(!is.na(padj))

  sig_genes <- res %>% filter(padj < 0.05, abs(log2FoldChange) > 0.5) %>% pull(gene)
  bg_genes  <- res %>% pull(gene)

  sig_entrez <- suppressMessages(
    bitr(sig_genes, fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))$ENTREZID
  bg_entrez  <- suppressMessages(
    bitr(bg_genes,  fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))$ENTREZID

  if (length(sig_entrez) < 10) {
    cat(sprintf("  %s: too few mapped genes (%d), skipping\n", ct_labels[ct], length(sig_entrez)))
    next
  }

  kegg <- tryCatch(
    enrichKEGG(gene=sig_entrez, universe=bg_entrez, organism="hsa",
               pAdjustMethod="BH", pvalueCutoff=0.05, qvalueCutoff=0.2,
               minGSSize=10, maxGSSize=500),
    error=function(e) { cat("  KEGG error:", conditionMessage(e), "\n"); NULL }
  )

  if (!is.null(kegg) && nrow(kegg@result) > 0) {
    n_sig <- sum(kegg@result$p.adjust < 0.05, na.rm=TRUE)
    cat(sprintf("  %-28s %d sig KEGG pathways (of %d tested)\n",
                ct_labels[ct], n_sig, nrow(kegg@result)))
    ora_results[[ct]] <- kegg
    # Save per-cell-type results
    write_csv(as.data.frame(kegg@result),
              sprintf("/mnt/results/08_pathways/kegg_ora_%s_severe_vs_healthy.csv", ct))
  } else {
    cat(sprintf("  %-28s no significant KEGG pathways\n", ct_labels[ct]))
  }
}

cat(sprintf("\nORA complete: %d/%d cell types with significant pathways\n",
            length(ora_results), length(cell_types)))


Top-20 DEGs table saved: 121 rows

=== KEGG ORA (severe vs healthy, padj<0.05, |LFC|>0.5) ===
  NP: canonical                0 sig KEGG pathways (of 318 tested)
  NP: HAPLN1+                  0 sig KEGG pathways (of 301 tested)
  NP: stress                   0 sig KEGG pathways (of 259 tested)
  NP: degenerative             2 sig KEGG pathways (of 306 tested)
  NP: MT-high                  0 sig KEGG pathways (of 223 tested)
  AF fibroblast                0 sig KEGG pathways (of 267 tested)

ORA complete: 6/6 cell types with significant pathwaysWarning: 70.36% of input gene IDs are fail to map...
Reading KEGG annotation online: "https://rest.kegg.jp/link/hsa/pathway"...

Reading KEGG annotation online: "https://rest.kegg.jp/list/pathway/hsa"...


Diagnose gene ID mapping failure and switch to MSigDB ORA with direct symbol matching

In [31]:

suppressPackageStartupMessages({
  library(dplyr)
  library(readr)
  library(clusterProfiler)
  library(org.Hs.eg.db)
  library(msigdbr)
  library(ggplot2)
  library(patchwork)
})
select <- dplyr::select

# ── Diagnose mapping failure ──────────────────────────────────────────────────
res_np <- read_csv("/workspace/data/pseudobulk/deseq2_NP_chondrocyte__severe_degen_vs_healthy.csv",
                   show_col_types=FALSE) %>% filter(!is.na(padj))

sig_genes <- res_np %>% filter(padj < 0.05, abs(log2FoldChange) > 0.5) %>% pull(gene)
cat(sprintf("Sig genes: %d\n", length(sig_genes)))
cat("Sample sig genes:", paste(head(sig_genes, 20), collapse=", "), "\n\n")

# Check what fraction are standard HGNC symbols
mapped <- suppressMessages(
  bitr(sig_genes, fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))
cat(sprintf("Mapped to Entrez: %d / %d (%.1f%%)\n",
            nrow(mapped), length(sig_genes), 100*nrow(mapped)/length(sig_genes)))

# Check unmapped genes
unmapped <- setdiff(sig_genes, mapped$SYMBOL)
cat("Sample unmapped:", paste(head(unmapped, 20), collapse=", "), "\n")

# Check if they look like non-standard names
cat("\nUnmapped gene patterns:\n")
cat("  Starting with MT-:", sum(grepl("^MT-", unmapped)), "\n")
cat("  Starting with RP[SL]:", sum(grepl("^RP[SL]", unmapped)), "\n")
cat("  Containing '.': ", sum(grepl("\\.", unmapped)), "\n")
cat("  All caps >6 chars:", sum(nchar(unmapped) > 6 & unmapped == toupper(unmapped)), "\n")


Sig genes: 2638
Sample sig genes: LINC01578, PFDN5, PYURF, CASC4, TAS2R20, H3F3B, PSMA2, IMPAD1, PRKAG1, WDR83OS, H3F3A, ZNRD1, FOPNL, H2AFZ, C12orf10, WDR34, SRD5A3, H2AFJ, H1FX, NUDT21 

Mapped to Entrez: 782 / 2638 (29.6%)
Sample unmapped: LINC01578, CASC4, H3F3B, IMPAD1, H3F3A, ZNRD1, FOPNL, H2AFZ, C12orf10, WDR34, H2AFJ, H1FX, TWISTNB, WDR60, ATP5MPL, HIST1H1C, TSTA3, ATP5MD, TCTEX1D2, KIAA0355 

Unmapped gene patterns:
  Starting with MT-: 0 
  Starting with RP[SL]: 1 
  Containing '.':  1587 
  All caps >6 chars: 1697 Warning: package 'msigdbr' was built under R version 4.4.3

Investigate dot-containing gene names and fix alias mapping

In [33]:

# 1587 genes contain dots — these are old-style aliases (e.g. H2AFZ, C12orf10)
# or Ensembl-style names that got dots from the MTX loading
dot_genes <- unmapped[grepl("\\.", unmapped)]
cat("Dot-containing unmapped genes (first 30):\n")
cat(paste(head(dot_genes, 30), collapse="\n"), "\n\n")

# Check non-dot unmapped
nodot_unmapped <- unmapped[!grepl("\\.", unmapped)]
cat(sprintf("Non-dot unmapped: %d\n", length(nodot_unmapped)))
cat("Sample:", paste(head(nodot_unmapped, 20), collapse=", "), "\n\n")

# These look like old HGNC aliases — try ALIAS mapping
alias_map <- suppressMessages(
  bitr(nodot_unmapped, fromType="ALIAS", toType=c("SYMBOL","ENTREZID"), OrgDb=org.Hs.eg.db))
cat(sprintf("Recovered via ALIAS mapping: %d / %d\n",
            length(unique(alias_map$ALIAS)), length(nodot_unmapped)))
cat("Examples:\n")
print(head(alias_map, 10))


Dot-containing unmapped genes (first 30):
AC118549.1
AC058791.1
AL592183.1
AC007325.4
AC093323.1
AL049839.2
AC104051.2
AC005165.1
AC007906.2
AC100810.1
AL162231.1
AP001816.1
AC020916.1
AC010642.2
AC091271.1
AC068888.1
AC027644.3
AL392172.1
AL118516.1
AC144652.1
AC245297.3
AC005261.1
AC060780.1
AC025159.1
AC093673.1
AC025181.2
BX284668.5
AL035258.1
AC016831.5
AC008124.1 

Non-dot unmapped: 269
Sample: LINC01578, CASC4, H3F3B, IMPAD1, H3F3A, ZNRD1, FOPNL, H2AFZ, C12orf10, WDR34, H2AFJ, H1FX, TWISTNB, WDR60, ATP5MPL, HIST1H1C, TSTA3, ATP5MD, TCTEX1D2, KIAA0355 

Recovered via ALIAS mapping: 265 / 269
Examples:
       ALIAS  SYMBOL  ENTREZID
1  LINC01578 CHASERR 100507217
2      CASC4   GOLM2    113201
3      H3F3B   H3-3B      3021
4     IMPAD1   BPNT2     54928
5      H3F3A   H3-3A      3020
6      ZNRD1  POLR1H     30834
7      FOPNL   CEP20    123811
8      H2AFZ   H2AZ1      3015
9   C12orf10    MYG1     60314
10     WDR34 DYNC2I2     89891
       ALIAS  SYMBOL  ENTREZID
1  LINC01578 

Run MSigDB ORA with symbol-based gene sets (Hallmarks + KEGG + Reactome) using alias-corrected gene names

In [35]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(clusterProfiler)
  library(org.Hs.eg.db); library(msigdbr)
  library(ggplot2); library(patchwork); library(tidyr)
})
select <- dplyr::select

# ── Build alias→current symbol lookup ────────────────────────────────────────
# Map old aliases to current HGNC symbols for the full gene universe
build_symbol_map <- function(genes) {
  # Try direct SYMBOL first
  direct <- suppressMessages(
    bitr(genes, fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))
  unmapped <- setdiff(genes, direct$SYMBOL)
  # Remove dot-containing (lncRNA/Ensembl IDs — not in MSigDB anyway)
  unmapped_clean <- unmapped[!grepl("\\.", unmapped)]
  # Try ALIAS
  via_alias <- suppressMessages(
    bitr(unmapped_clean, fromType="ALIAS", toType=c("SYMBOL","ENTREZID"),
         OrgDb=org.Hs.eg.db)) %>%
    rename(original=ALIAS)
  # Build lookup: original → current symbol
  lookup <- c(setNames(direct$SYMBOL, direct$SYMBOL),
              setNames(via_alias$SYMBOL, via_alias$original))
  lookup
}

# ── Load MSigDB gene sets (Hallmarks + KEGG + Reactome) ──────────────────────
cat("Loading MSigDB gene sets...\n")
h_sets  <- msigdbr(species="Homo sapiens", category="H") %>%
  dplyr::select(gs_name, gene_symbol)
c2_kegg <- msigdbr(species="Homo sapiens", category="C2", subcategory="CP:KEGG") %>%
  dplyr::select(gs_name, gene_symbol)
c2_reac <- msigdbr(species="Homo sapiens", category="C2", subcategory="CP:REACTOME") %>%
  dplyr::select(gs_name, gene_symbol)

cat(sprintf("  Hallmarks: %d gene sets\n", n_distinct(h_sets$gs_name)))
cat(sprintf("  KEGG:      %d gene sets\n", n_distinct(c2_kegg$gs_name)))
cat(sprintf("  Reactome:  %d gene sets\n", n_distinct(c2_reac$gs_name)))

# Combine all
all_sets <- bind_rows(
  h_sets %>% mutate(collection="Hallmarks"),
  c2_kegg %>% mutate(collection="KEGG"),
  c2_reac %>% mutate(collection="Reactome")
)

# ── ORA per cell type using enricher() (symbol-based) ────────────────────────
cell_types <- c("NP_chondrocyte","NP_chondrocyte_HAPLN1","NP_stress_response",
                "NP_degenerative_UPR","NP_metallothionein","AF_fibroblast")
ct_labels  <- c("NP_chondrocyte"="NP: canonical","NP_chondrocyte_HAPLN1"="NP: HAPLN1+",
                "NP_stress_response"="NP: stress","NP_degenerative_UPR"="NP: degenerative",
                "NP_metallothionein"="NP: MT-high","AF_fibroblast"="AF fibroblast")

ora_all <- list()
cat("\n=== MSigDB ORA (severe vs healthy) ===\n")

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  res <- read_csv(f, show_col_types=FALSE) %>% filter(!is.na(padj))

  # Build symbol map for this cell type's genes
  sym_map <- build_symbol_map(res$gene)

  # Map gene names
  res$gene_mapped <- sym_map[res$gene]

  sig_genes <- res %>%
    filter(padj < 0.05, abs(log2FoldChange) > 0.5, !is.na(gene_mapped)) %>%
    pull(gene_mapped) %>% unique()
  bg_genes <- res %>% filter(!is.na(gene_mapped)) %>% pull(gene_mapped) %>% unique()

  cat(sprintf("  %-28s sig=%d  bg=%d\n", ct_labels[ct], length(sig_genes), length(bg_genes)))

  if (length(sig_genes) < 10) next

  # Run enricher with all MSigDB sets
  enr <- tryCatch(
    enricher(gene=sig_genes, universe=bg_genes,
             TERM2GENE=dplyr::select(all_sets, gs_name, gene_symbol),
             TERM2NAME=NULL,
             pAdjustMethod="BH", pvalueCutoff=0.05, qvalueCutoff=0.2,
             minGSSize=10, maxGSSize=500),
    error=function(e) { cat("    Error:", conditionMessage(e), "\n"); NULL }
  )

  if (!is.null(enr) && nrow(enr@result) > 0) {
    res_df <- as.data.frame(enr@result) %>%
      filter(p.adjust < 0.05) %>%
      left_join(distinct(all_sets, gs_name, collection), by=c("ID"="gs_name")) %>%
      arrange(p.adjust)
    n_sig <- nrow(res_df)
    cat(sprintf("    → %d sig pathways (%s)\n", n_sig,
                paste(table(res_df$collection), collapse=" / ")))
    ora_all[[ct]] <- res_df
    write_csv(res_df, sprintf("/mnt/results/08_pathways/msigdb_ora_%s_severe_vs_healthy.csv", ct))
  } else {
    cat("    → no significant pathways\n")
  }
}

cat(sprintf("\nDone: %d/%d cell types with significant pathways\n",
            length(ora_all), length(cell_types)))


Loading MSigDB gene sets...
Error: Unknown subcollection. Warning: package 'tidyr' was built under R version 4.4.3
i Please use the `collection` argument instead.
i Please use the `subcollection` argument instead.

Fix msigdbr v10 API and rerun MSigDB ORA

In [37]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(clusterProfiler)
  library(org.Hs.eg.db); library(msigdbr)
  library(ggplot2); library(patchwork)
})
select <- dplyr::select

# Check msigdbr v10 API
cat("msigdbr version:", as.character(packageVersion("msigdbr")), "\n")
# List available collections
avail <- msigdbr_collections()
print(avail)


msigdbr version: 25.1.1 
# A tibble: 25 x 4
   gs_collection gs_subcollection  gs_collection_name                 num_genesets
   <chr>         <chr>             <chr>                                     <int>
 1 C1            ""                Positional                                  302
 2 C2            "CGP"             Chemical and Genetic Perturbations         3538
 3 C2            "CP"              Canonical Pathways                           19
 4 C2            "CP:BIOCARTA"     BioCarta Pathways                           292
 5 C2            "CP:KEGG_LEGACY"  KEGG Legacy Pathways                        186
 6 C2            "CP:KEGG_MEDICUS" KEGG Medicus Pathways                       658
 7 C2            "CP:PID"          PID Pathways                                196
 8 C2            "CP:REACTOME"     Reactome Pathways                          1787
 9 C2            "CP:WIKIPATHWAYS" WikiPathways                                885
10 C3            "MIR:MIRDB"       miRDB   

Load MSigDB gene sets with correct v10 API and run ORA across all cell types

In [39]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(clusterProfiler)
  library(org.Hs.eg.db); library(msigdbr)
  library(ggplot2); library(patchwork)
})
select <- dplyr::select

# ── Load gene sets with v10 API ───────────────────────────────────────────────
cat("Loading MSigDB gene sets (v10 API)...\n")
h_sets  <- msigdbr(species="Homo sapiens", collection="H") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="Hallmarks")
kegg_sets <- msigdbr(species="Homo sapiens", collection="C2", subcollection="CP:KEGG_LEGACY") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="KEGG")
reac_sets <- msigdbr(species="Homo sapiens", collection="C2", subcollection="CP:REACTOME") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="Reactome")

all_sets <- bind_rows(h_sets, kegg_sets, reac_sets)
cat(sprintf("  Hallmarks: %d | KEGG: %d | Reactome: %d gene sets\n",
            n_distinct(h_sets$gs_name), n_distinct(kegg_sets$gs_name), n_distinct(reac_sets$gs_name)))

# ── Alias lookup helper ───────────────────────────────────────────────────────
build_symbol_map <- function(genes) {
  direct <- suppressMessages(
    bitr(genes, fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))
  unmapped <- setdiff(genes, direct$SYMBOL)
  unmapped_clean <- unmapped[!grepl("\\.", unmapped)]
  via_alias <- suppressMessages(
    bitr(unmapped_clean, fromType="ALIAS", toType=c("SYMBOL","ENTREZID"),
         OrgDb=org.Hs.eg.db)) %>% rename(original=ALIAS)
  c(setNames(direct$SYMBOL, direct$SYMBOL),
    setNames(via_alias$SYMBOL, via_alias$original))
}

# ── ORA per cell type ─────────────────────────────────────────────────────────
cell_types <- c("NP_chondrocyte","NP_chondrocyte_HAPLN1","NP_stress_response",
                "NP_degenerative_UPR","NP_metallothionein","AF_fibroblast")
ct_labels  <- c("NP_chondrocyte"="NP: canonical","NP_chondrocyte_HAPLN1"="NP: HAPLN1+",
                "NP_stress_response"="NP: stress","NP_degenerative_UPR"="NP: degenerative",
                "NP_metallothionein"="NP: MT-high","AF_fibroblast"="AF fibroblast")

ora_all <- list()
cat("\n=== MSigDB ORA (severe vs healthy, padj<0.05, |LFC|>0.5) ===\n")

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  res <- read_csv(f, show_col_types=FALSE) %>% filter(!is.na(padj))

  sym_map  <- build_symbol_map(res$gene)
  res$gene_mapped <- sym_map[res$gene]

  sig_genes <- res %>%
    filter(padj < 0.05, abs(log2FoldChange) > 0.5, !is.na(gene_mapped)) %>%
    pull(gene_mapped) %>% unique()
  bg_genes  <- res %>% filter(!is.na(gene_mapped)) %>% pull(gene_mapped) %>% unique()

  cat(sprintf("  %-28s sig=%d  bg=%d\n", ct_labels[ct], length(sig_genes), length(bg_genes)))
  if (length(sig_genes) < 10) next

  enr <- tryCatch(
    enricher(gene=sig_genes, universe=bg_genes,
             TERM2GENE=dplyr::select(all_sets, gs_name, gene_symbol),
             pAdjustMethod="BH", pvalueCutoff=0.05, qvalueCutoff=0.2,
             minGSSize=10, maxGSSize=500),
    error=function(e) { cat("    Error:", conditionMessage(e), "\n"); NULL }
  )

  if (!is.null(enr) && nrow(enr@result) > 0) {
    res_df <- as.data.frame(enr@result) %>%
      filter(p.adjust < 0.05) %>%
      left_join(distinct(all_sets, gs_name, collection), by=c("ID"="gs_name")) %>%
      arrange(p.adjust)
    cat(sprintf("    → %d sig pathways | Hallmarks=%d KEGG=%d Reactome=%d\n",
                nrow(res_df),
                sum(res_df$collection=="Hallmarks", na.rm=TRUE),
                sum(res_df$collection=="KEGG",      na.rm=TRUE),
                sum(res_df$collection=="Reactome",  na.rm=TRUE)))
    ora_all[[ct]] <- res_df
    write_csv(res_df,
      sprintf("/mnt/results/08_pathways/msigdb_ora_%s_severe_vs_healthy.csv", ct))
  } else {
    cat("    → no significant pathways\n")
  }
}

cat(sprintf("\nDone: %d/%d cell types with results\n", length(ora_all), length(cell_types)))


Loading MSigDB gene sets (v10 API)...
  Hallmarks: 50 | KEGG: 186 | Reactome: 1787 gene sets

=== MSigDB ORA (severe vs healthy, padj<0.05, |LFC|>0.5) ===
  NP: canonical                sig=1043  bg=15002
    <e2><86><92> 94 sig pathways | Hallmarks=5 KEGG=3 Reactome=86
  NP: HAPLN1+                  sig=866  bg=13806
    <e2><86><92> 94 sig pathways | Hallmarks=1 KEGG=3 Reactome=90
  NP: stress                   sig=549  bg=12165
    <e2><86><92> 83 sig pathways | Hallmarks=1 KEGG=2 Reactome=80
  NP: degenerative             sig=938  bg=14542
    <e2><86><92> 95 sig pathways | Hallmarks=1 KEGG=3 Reactome=91
  NP: MT-high                  sig=407  bg=12787
    <e2><86><92> 94 sig pathways | Hallmarks=1 KEGG=3 Reactome=90
  AF fibroblast                sig=600  bg=13829
    <e2><86><92> 91 sig pathways | Hallmarks=0 KEGG=3 Reactome=88

Done: 6/6 cell types with resultsWarning: 13.76% of input gene IDs are fail to map...

Generate pathway heatmap: top Hallmark pathways across all cell types

In [41]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(tidyr)
  library(ggplot2); library(patchwork)
})
select <- dplyr::select

cell_types <- c("NP_chondrocyte","NP_chondrocyte_HAPLN1","NP_stress_response",
                "NP_degenerative_UPR","NP_metallothionein","AF_fibroblast")
ct_labels  <- c("NP_chondrocyte"="NP: canonical","NP_chondrocyte_HAPLN1"="NP: HAPLN1+",
                "NP_stress_response"="NP: stress","NP_degenerative_UPR"="NP: degenerative",
                "NP_metallothionein"="NP: MT-high","AF_fibroblast"="AF fibroblast")

# ── Load all ORA results ──────────────────────────────────────────────────────
all_ora <- lapply(cell_types, function(ct) {
  f <- sprintf("/mnt/results/08_pathways/msigdb_ora_%s_severe_vs_healthy.csv", ct)
  if (!file.exists(f)) return(NULL)
  read_csv(f, show_col_types=FALSE) %>% mutate(cell_type=ct)
}) %>% bind_rows()

cat(sprintf("Total significant pathway-celltype pairs: %d\n", nrow(all_ora)))
cat(sprintf("Collections: %s\n", paste(table(all_ora$collection), collapse=" / ")))

# ── Panel A: Hallmark dotplot across cell types ───────────────────────────────
hallmarks <- all_ora %>%
  filter(collection == "Hallmarks") %>%
  mutate(
    pathway_short = gsub("HALLMARK_", "", ID),
    pathway_short = gsub("_", " ", pathway_short),
    pathway_short = tools::toTitleCase(tolower(pathway_short)),
    cell_label    = ct_labels[cell_type],
    GeneRatio_num = sapply(GeneRatio, function(x) {
      parts <- strsplit(x, "/")[[1]]; as.numeric(parts[1])/as.numeric(parts[2])
    }),
    neg_log10_padj = pmin(-log10(p.adjust), 10)
  )

# Get union of Hallmarks significant in any cell type
sig_hallmarks <- hallmarks %>%
  group_by(pathway_short) %>%
  summarise(min_padj=min(p.adjust), n_cts=n_distinct(cell_type)) %>%
  arrange(min_padj) %>%
  pull(pathway_short)

cat(sprintf("\nSignificant Hallmarks: %d\n", length(sig_hallmarks)))
print(sig_hallmarks)

# Full matrix (fill missing with NA)
ct_order <- c("NP: canonical","NP: HAPLN1+","NP: stress",
              "NP: degenerative","NP: MT-high","AF fibroblast")

hallmarks_plot <- hallmarks %>%
  filter(pathway_short %in% sig_hallmarks) %>%
  mutate(cell_label = factor(cell_label, levels=ct_order),
         pathway_short = factor(pathway_short, levels=rev(sig_hallmarks)))

p_hallmarks <- ggplot(hallmarks_plot,
       aes(x=cell_label, y=pathway_short,
           size=GeneRatio_num, color=neg_log10_padj)) +
  geom_point() +
  scale_size_continuous(range=c(2,8), name="Gene ratio") +
  scale_color_gradient(low="#FEE090", high="#A50026",
                       name="-log10(padj)", limits=c(0,10)) +
  labs(title="Hallmark pathways enriched in severe degeneration",
       subtitle="ORA: severe vs healthy | MSigDB Hallmarks",
       x=NULL, y=NULL) +
  theme_classic(base_size=10) +
  theme(axis.text.x=element_text(angle=35, hjust=1, size=9),
        axis.text.y=element_text(size=9),
        plot.title=element_text(face="bold", size=11),
        legend.position="right",
        panel.grid.major=element_line(color="gray92", linewidth=0.4))

# ── Panel B: Top Reactome pathways (shared across ≥3 cell types) ─────────────
reactome_shared <- all_ora %>%
  filter(collection == "Reactome") %>%
  group_by(ID) %>%
  summarise(n_cts=n_distinct(cell_type), min_padj=min(p.adjust)) %>%
  filter(n_cts >= 3) %>%
  arrange(desc(n_cts), min_padj) %>%
  head(25) %>%
  pull(ID)

cat(sprintf("\nReactome pathways in ≥3 cell types: %d (showing top 25)\n",
            length(reactome_shared)))

reactome_plot <- all_ora %>%
  filter(collection=="Reactome", ID %in% reactome_shared) %>%
  mutate(
    pathway_short = gsub("REACTOME_", "", ID),
    pathway_short = gsub("_", " ", pathway_short),
    pathway_short = tools::toTitleCase(tolower(pathway_short)),
    pathway_short = substr(pathway_short, 1, 55),  # truncate long names
    cell_label    = ct_labels[cell_type],
    GeneRatio_num = sapply(GeneRatio, function(x) {
      parts <- strsplit(x, "/")[[1]]; as.numeric(parts[1])/as.numeric(parts[2])
    }),
    neg_log10_padj = pmin(-log10(p.adjust), 10)
  ) %>%
  mutate(cell_label = factor(cell_label, levels=ct_order))

# Order pathways by mean -log10(padj)
pw_order <- reactome_plot %>%
  group_by(pathway_short) %>%
  summarise(mean_sig=mean(neg_log10_padj)) %>%
  arrange(mean_sig) %>% pull(pathway_short)
reactome_plot$pathway_short <- factor(reactome_plot$pathway_short, levels=pw_order)

p_reactome <- ggplot(reactome_plot,
       aes(x=cell_label, y=pathway_short,
           size=GeneRatio_num, color=neg_log10_padj)) +
  geom_point() +
  scale_size_continuous(range=c(2,7), name="Gene ratio") +
  scale_color_gradient(low="#FEE090", high="#A50026",
                       name="-log10(padj)", limits=c(0,10)) +
  labs(title="Shared Reactome pathways (≥3 cell types)",
       subtitle="ORA: severe vs healthy | MSigDB Reactome",
       x=NULL, y=NULL) +
  theme_classic(base_size=9) +
  theme(axis.text.x=element_text(angle=35, hjust=1, size=8.5),
        axis.text.y=element_text(size=8),
        plot.title=element_text(face="bold", size=10),
        legend.position="right",
        panel.grid.major=element_line(color="gray92", linewidth=0.4))

# ── Combine and save ──────────────────────────────────────────────────────────
combined <- p_hallmarks / p_reactome +
  plot_layout(heights=c(1, 2.2)) +
  plot_annotation(
    title="Pathway enrichment: severe IVD degeneration vs healthy",
    theme=theme(plot.title=element_text(face="bold", size=13))
  )

ggsave("/mnt/results/08_pathways/pathway_dotplot_severe_vs_healthy.png",
       combined, width=13, height=16, dpi=150)
cat("Saved pathway_dotplot_severe_vs_healthy.png\n")


Total significant pathway-celltype pairs: 551
Collections: 9 / 17 / 525

Significant Hallmarks: 5
[1] "Glycolysis"                        "Tnfa Signaling via Nfkb"           "Epithelial Mesenchymal Transition"
[4] "Hypoxia"                           "Kras Signaling Up"                

Reactome pathways in <e2><89><a5>3 cell types: 25 (showing top 25)
Saved pathway_dotplot_severe_vs_healthy.png

Run GSEA on ranked gene lists + filter Reactome to IVD-relevant categories for better specificity

In [43]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(tidyr)
  library(clusterProfiler); library(org.Hs.eg.db); library(msigdbr)
  library(ggplot2); library(patchwork)
})
select <- dplyr::select

# ── Gene sets ─────────────────────────────────────────────────────────────────
h_sets    <- msigdbr(species="Homo sapiens", collection="H") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="Hallmarks")
reac_sets <- msigdbr(species="Homo sapiens", collection="C2", subcollection="CP:REACTOME") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="Reactome")

# Filter Reactome to IVD-relevant categories (exclude cell cycle, meiosis, etc.)
exclude_patterns <- c("CELL_CYCLE","MITOTIC","MEIOTIC","MEIOSIS","SPERMATOGENESIS",
                      "OOCYTE","MATERNAL_TO_ZYGOTIC","PROTAMINE","TELOMERE_MAINTENANCE",
                      "DNA_REPLICATION","CHROMOSOME_MAINTENANCE","CHROMATIN_MODIFYING",
                      "HISTONE_MODIFICATIONS","MRNA_PROCESSING","RRNA","TRNA",
                      "TRANSLATION","RIBOSOM","NONSENSE","MRNA_SPLICING","MRNA_DECAY",
                      "TRANSCRIPTIONAL_REGULATION_BY_SMALL_RNA","RETROTRANSPOSON",
                      "ENDOGENOUS_RETROELEMENT","GRANULOPOIESIS","MEGAKARYOCYTE",
                      "ANDROGEN","ERYTHROID","PLATELET","NEUTROPHIL_DEGRANULATION")
excl_regex <- paste(exclude_patterns, collapse="|")

reac_ivd <- reac_sets %>%
  filter(!grepl(excl_regex, gs_name))
cat(sprintf("Reactome sets: %d total → %d after IVD-relevant filter\n",
            n_distinct(reac_sets$gs_name), n_distinct(reac_ivd$gs_name)))

all_sets <- bind_rows(h_sets, reac_ivd)

# ── Alias lookup ──────────────────────────────────────────────────────────────
build_symbol_map <- function(genes) {
  direct <- suppressMessages(
    bitr(genes, fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))
  unmapped <- setdiff(genes, direct$SYMBOL)
  unmapped_clean <- unmapped[!grepl("\\.", unmapped)]
  via_alias <- suppressMessages(
    bitr(unmapped_clean, fromType="ALIAS", toType=c("SYMBOL","ENTREZID"),
         OrgDb=org.Hs.eg.db)) %>% rename(original=ALIAS)
  c(setNames(direct$SYMBOL, direct$SYMBOL),
    setNames(via_alias$SYMBOL, via_alias$original))
}

# ── GSEA per cell type ────────────────────────────────────────────────────────
cell_types <- c("NP_chondrocyte","NP_chondrocyte_HAPLN1","NP_stress_response",
                "NP_degenerative_UPR","NP_metallothionein","AF_fibroblast")
ct_labels  <- c("NP_chondrocyte"="NP: canonical","NP_chondrocyte_HAPLN1"="NP: HAPLN1+",
                "NP_stress_response"="NP: stress","NP_degenerative_UPR"="NP: degenerative",
                "NP_metallothionein"="NP: MT-high","AF_fibroblast"="AF fibroblast")

gsea_all <- list()
cat("\n=== GSEA (severe vs healthy, ranked by sign(LFC)*-log10(padj)) ===\n")

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  res <- read_csv(f, show_col_types=FALSE) %>%
    filter(!is.na(padj), !is.na(log2FoldChange))

  sym_map <- build_symbol_map(res$gene)
  res$gene_mapped <- sym_map[res$gene]
  res <- res %>% filter(!is.na(gene_mapped))

  # Rank metric: sign(LFC) * -log10(padj), capped
  res <- res %>%
    mutate(
      padj_safe = pmax(padj, 1e-300),
      rank_metric = sign(log2FoldChange) * (-log10(padj_safe))
    ) %>%
    arrange(desc(rank_metric)) %>%
    # Keep one entry per mapped symbol (highest |rank|)
    group_by(gene_mapped) %>%
    slice_max(abs(rank_metric), n=1) %>%
    ungroup()

  gene_list <- setNames(res$rank_metric, res$gene_mapped)
  cat(sprintf("  %-28s n=%d genes ranked\n", ct_labels[ct], length(gene_list)))

  gsea <- tryCatch(
    GSEA(geneList=gene_list,
         TERM2GENE=dplyr::select(all_sets, gs_name, gene_symbol),
         pAdjustMethod="BH", pvalueCutoff=0.05,
         minGSSize=10, maxGSSize=500,
         eps=0, seed=42, verbose=FALSE),
    error=function(e) { cat("    GSEA error:", conditionMessage(e), "\n"); NULL }
  )

  if (!is.null(gsea) && nrow(gsea@result) > 0) {
    res_df <- as.data.frame(gsea@result) %>%
      filter(p.adjust < 0.05) %>%
      left_join(distinct(all_sets, gs_name, collection), by=c("ID"="gs_name")) %>%
      arrange(p.adjust)
    n_up <- sum(res_df$NES > 0); n_dn <- sum(res_df$NES < 0)
    cat(sprintf("    → %d sig (NES+=%d, NES-=%d) | H=%d R=%d\n",
                nrow(res_df), n_up, n_dn,
                sum(res_df$collection=="Hallmarks",na.rm=TRUE),
                sum(res_df$collection=="Reactome",na.rm=TRUE)))
    gsea_all[[ct]] <- res_df
    write_csv(res_df,
      sprintf("/mnt/results/08_pathways/gsea_%s_severe_vs_healthy.csv", ct))
  } else {
    cat("    → no significant pathways\n")
  }
}

cat(sprintf("\nGSEA done: %d/%d cell types with results\n",
            length(gsea_all), length(cell_types)))


Reactome sets: 1787 total <e2><86><92> 1679 after IVD-relevant filter

=== GSEA (severe vs healthy, ranked by sign(LFC)*-log10(padj)) ===
  NP: canonical                n=15002 genes ranked
    GSEA error: geneList should be a decreasing sorted vector... 
    <e2><86><92> no significant pathways
  NP: HAPLN1+                  n=13806 genes ranked
    GSEA error: geneList should be a decreasing sorted vector... 
    <e2><86><92> no significant pathways
  NP: stress                   n=12165 genes ranked
    GSEA error: geneList should be a decreasing sorted vector... 
    <e2><86><92> no significant pathways
  NP: degenerative             n=14542 genes ranked
    GSEA error: geneList should be a decreasing sorted vector... 
    <e2><86><92> no significant pathways
  NP: MT-high                  n=12787 genes ranked
    GSEA error: geneList should be a decreasing sorted vector... 
    <e2><86><92> no significant pathways
  AF fibroblast                n=13829 genes ranked
    GSEA error:

Fix GSEA: ensure gene list is properly sorted descending before passing to GSEA

In [45]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(clusterProfiler)
  library(org.Hs.eg.db); library(msigdbr); library(ggplot2)
})
select <- dplyr::select

# Reload gene sets
h_sets    <- msigdbr(species="Homo sapiens", collection="H") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="Hallmarks")
reac_sets <- msigdbr(species="Homo sapiens", collection="C2", subcollection="CP:REACTOME") %>%
  dplyr::select(gs_name, gene_symbol) %>% mutate(collection="Reactome")
exclude_patterns <- c("CELL_CYCLE","MITOTIC","MEIOTIC","MEIOSIS","SPERMATOGENESIS",
                      "OOCYTE","MATERNAL_TO_ZYGOTIC","PROTAMINE","TELOMERE_MAINTENANCE",
                      "DNA_REPLICATION","CHROMOSOME_MAINTENANCE","CHROMATIN_MODIFYING",
                      "HISTONE_MODIFICATIONS","MRNA_PROCESSING","RRNA","TRNA",
                      "TRANSLATION","RIBOSOM","NONSENSE","MRNA_SPLICING","MRNA_DECAY",
                      "RETROTRANSPOSON","ENDOGENOUS_RETROELEMENT","GRANULOPOIESIS",
                      "MEGAKARYOCYTE","ANDROGEN","ERYTHROID","PLATELET")
reac_ivd <- reac_sets %>% filter(!grepl(paste(exclude_patterns,collapse="|"), gs_name))
all_sets  <- bind_rows(h_sets, reac_ivd)

build_symbol_map <- function(genes) {
  direct <- suppressMessages(bitr(genes, fromType="SYMBOL", toType="ENTREZID", OrgDb=org.Hs.eg.db))
  unmapped <- setdiff(genes, direct$SYMBOL)
  unmapped_clean <- unmapped[!grepl("\\.", unmapped)]
  via_alias <- suppressMessages(
    bitr(unmapped_clean, fromType="ALIAS", toType=c("SYMBOL","ENTREZID"), OrgDb=org.Hs.eg.db)) %>%
    rename(original=ALIAS)
  c(setNames(direct$SYMBOL, direct$SYMBOL), setNames(via_alias$SYMBOL, via_alias$original))
}

cell_types <- c("NP_chondrocyte","NP_chondrocyte_HAPLN1","NP_stress_response",
                "NP_degenerative_UPR","NP_metallothionein","AF_fibroblast")
ct_labels  <- c("NP_chondrocyte"="NP: canonical","NP_chondrocyte_HAPLN1"="NP: HAPLN1+",
                "NP_stress_response"="NP: stress","NP_degenerative_UPR"="NP: degenerative",
                "NP_metallothionein"="NP: MT-high","AF_fibroblast"="AF fibroblast")

gsea_all <- list()
cat("=== GSEA (severe vs healthy) ===\n")

for (ct in cell_types) {
  f <- sprintf("/workspace/data/pseudobulk/deseq2_%s__severe_degen_vs_healthy.csv", ct)
  if (!file.exists(f)) next
  res <- read_csv(f, show_col_types=FALSE) %>%
    filter(!is.na(padj), !is.na(log2FoldChange))

  sym_map <- build_symbol_map(res$gene)
  res$gene_mapped <- sym_map[res$gene]
  res <- res %>% filter(!is.na(gene_mapped)) %>%
    mutate(padj_safe = pmax(padj, 1e-300),
           rank_metric = sign(log2FoldChange) * (-log10(padj_safe))) %>%
    group_by(gene_mapped) %>%
    slice_max(abs(rank_metric), n=1, with_ties=FALSE) %>%
    ungroup() %>%
    arrange(desc(rank_metric))  # MUST be sorted descending

  # Build named vector — already sorted
  gene_list <- setNames(res$rank_metric, res$gene_mapped)
  
  # Verify sorting
  stopifnot(all(diff(gene_list) <= 0))
  cat(sprintf("  %-28s n=%d | top: %s (%.1f), bottom: %s (%.1f)\n",
              ct_labels[ct], length(gene_list),
              names(gene_list)[1], gene_list[1],
              names(gene_list)[length(gene_list)], gene_list[length(gene_list)]))

  gsea <- tryCatch(
    GSEA(geneList=gene_list,
         TERM2GENE=dplyr::select(all_sets, gs_name, gene_symbol),
         pAdjustMethod="BH", pvalueCutoff=0.05,
         minGSSize=10, maxGSSize=500,
         eps=0, seed=42, verbose=FALSE),
    error=function(e) { cat("    Error:", conditionMessage(e), "\n"); NULL }
  )

  if (!is.null(gsea) && nrow(gsea@result) > 0) {
    res_df <- as.data.frame(gsea@result) %>%
      filter(p.adjust < 0.05) %>%
      left_join(distinct(all_sets, gs_name, collection), by=c("ID"="gs_name")) %>%
      arrange(NES)
    n_up <- sum(res_df$NES > 0); n_dn <- sum(res_df$NES < 0)
    cat(sprintf("    → %d sig | NES+=%d NES-=%d | H=%d R=%d\n",
                nrow(res_df), n_up, n_dn,
                sum(res_df$collection=="Hallmarks",na.rm=TRUE),
                sum(res_df$collection=="Reactome",na.rm=TRUE)))
    gsea_all[[ct]] <- res_df
    write_csv(res_df, sprintf("/mnt/results/08_pathways/gsea_%s_severe_vs_healthy.csv", ct))
  } else {
    cat("    → no significant pathways\n")
  }
}

cat(sprintf("\nGSEA done: %d/%d cell types\n", length(gsea_all), length(cell_types)))


=== GSEA (severe vs healthy) ===
  NP: canonical                n=15002 | top: TAS2R20 (21.4), bottom: CHASERR (-30.9)
    <e2><86><92> 85 sig | NES+=11 NES-=74 | H=1 R=84
  NP: HAPLN1+                  n=13806 | top: NUDT4B (13.4), bottom: CHASERR (-38.6)
    <e2><86><92> 73 sig | NES+=3 NES-=70 | H=0 R=73
  NP: stress                   n=12165 | top: P2RY6 (6.4), bottom: PFDN5 (-33.5)
    <e2><86><92> 70 sig | NES+=4 NES-=66 | H=2 R=68
  NP: degenerative             n=14542 | top: POMK (10.9), bottom: CHASERR (-31.9)
    <e2><86><92> 77 sig | NES+=8 NES-=69 | H=4 R=73
  NP: MT-high                  n=12787 | top: ZNF295-AS1 (4.4), bottom: PFDN5 (-23.2)
    <e2><86><92> 72 sig | NES+=6 NES-=66 | H=0 R=72
  AF fibroblast                n=13829 | top: NUDT4B (8.5), bottom: CHASERR (-34.2)
    <e2><86><92> 80 sig | NES+=11 NES-=69 | H=0 R=80

GSEA done: 6/6 cell typesWarning: 13.76% of input gene IDs are fail to map...
The order of those tied genes will be arbitrary, which may produce un

Generate GSEA summary heatmap: top pathways by NES across all cell types

In [47]:

suppressPackageStartupMessages({
  library(dplyr); library(readr); library(tidyr)
  library(ggplot2); library(patchwork)
})
select <- dplyr::select

cell_types <- c("NP_chondrocyte","NP_chondrocyte_HAPLN1","NP_stress_response",
                "NP_degenerative_UPR","NP_metallothionein","AF_fibroblast")
ct_labels  <- c("NP_chondrocyte"="NP: canonical","NP_chondrocyte_HAPLN1"="NP: HAPLN1+",
                "NP_stress_response"="NP: stress","NP_degenerative_UPR"="NP: degenerative",
                "NP_metallothionein"="NP: MT-high","AF_fibroblast"="AF fibroblast")
ct_order   <- c("NP: canonical","NP: HAPLN1+","NP: stress",
                "NP: degenerative","NP: MT-high","AF fibroblast")

# Load all GSEA results
gsea_all <- lapply(cell_types, function(ct) {
  f <- sprintf("/mnt/results/08_pathways/gsea_%s_severe_vs_healthy.csv", ct)
  if (!file.exists(f)) return(NULL)
  read_csv(f, show_col_types=FALSE) %>%
    mutate(cell_type=ct, cell_label=ct_labels[ct])
}) %>% bind_rows()

cat(sprintf("Total GSEA results: %d pathway×celltype pairs\n", nrow(gsea_all)))

# ── Select top pathways for visualization ─────────────────────────────────────
# Strategy: pick pathways with strongest NES (up or down) in ≥2 cell types
# Focus on Hallmarks + biologically interpretable Reactome

# Hallmarks: all significant ones
hallmark_paths <- gsea_all %>%
  filter(collection == "Hallmarks", p.adjust < 0.05) %>%
  group_by(ID) %>%
  summarise(n_cts=n_distinct(cell_type), mean_NES=mean(NES, na.rm=TRUE)) %>%
  arrange(mean_NES) %>% pull(ID)

cat(sprintf("Significant Hallmarks: %d\n", length(hallmark_paths)))
print(hallmark_paths)

# Reactome: IVD-relevant pathways with strongest signal in ≥2 cell types
# Key IVD biology keywords
ivd_keywords <- c("EXTRACELLULAR_MATRIX","COLLAGEN","INTEGRIN","TGF","WNT","NOTCH",
                  "HIPPO","VEGF","PDGF","FGF","MAPK","PI3K","MTOR","AUTOPHAGY",
                  "APOPTOSIS","SENESCENCE","OXIDATIVE","REACTIVE_OXYGEN","HYPOXIA",
                  "INFLAMMATION","INTERLEUKIN","INTERFERON","NF_KB","TNF","CHEMOKINE",
                  "IMMUNE","INNATE","ADAPTIVE","COMPLEMENT","COAGULATION",
                  "GLYCOLYSIS","METABOLISM","FATTY_ACID","LIPID","AMINO_ACID",
                  "PROTEOGLYCAN","AGGRECAN","VERSICAN","HEPARAN","CHONDROITIN",
                  "HEDGEHOG","BMP","SMAD","RUNX","SOX","FOXO","P53","CDKN",
                  "ANGIOGENESIS","BLOOD_VESSEL","ENDOTHELI","PERICYTE",
                  "CALCIUM","CXCR","CCR","CYTOKINE","GROWTH_FACTOR")
ivd_regex <- paste(ivd_keywords, collapse="|")

reactome_top <- gsea_all %>%
  filter(collection == "Reactome", p.adjust < 0.05,
         grepl(ivd_regex, ID)) %>%
  group_by(ID) %>%
  summarise(n_cts=n_distinct(cell_type), mean_NES=mean(NES, na.rm=TRUE),
            min_padj=min(p.adjust, na.rm=TRUE)) %>%
  filter(n_cts >= 2) %>%
  arrange(mean_NES) %>%
  # Take top 20 most downregulated + top 10 most upregulated
  { bind_rows(head(., 20), tail(., 10)) } %>%
  pull(ID)

cat(sprintf("\nSelected Reactome pathways (IVD-relevant, ≥2 CTs): %d\n", length(reactome_top)))

# ── Build NES matrix for heatmap ──────────────────────────────────────────────
selected_paths <- c(hallmark_paths, reactome_top)

nes_matrix <- gsea_all %>%
  filter(ID %in% selected_paths) %>%
  mutate(
    pathway_short = gsub("HALLMARK_|REACTOME_", "", ID),
    pathway_short = gsub("_", " ", pathway_short),
    pathway_short = tools::toTitleCase(tolower(pathway_short)),
    pathway_short = substr(pathway_short, 1, 60),
    cell_label    = factor(cell_label, levels=ct_order),
    sig_label     = ifelse(p.adjust < 0.05, "*", "")
  )

# Order pathways by mean NES
path_order <- nes_matrix %>%
  group_by(pathway_short) %>%
  summarise(mean_NES=mean(NES, na.rm=TRUE)) %>%
  arrange(mean_NES) %>% pull(pathway_short)
nes_matrix$pathway_short <- factor(nes_matrix$pathway_short, levels=path_order)

# ── Plot heatmap ──────────────────────────────────────────────────────────────
p <- ggplot(nes_matrix, aes(x=cell_label, y=pathway_short, fill=NES)) +
  geom_tile(color="white", linewidth=0.3) +
  geom_text(aes(label=sig_label), size=4, vjust=0.75, color="black") +
  scale_fill_gradient2(
    low="#2166AC", mid="white", high="#D6604D",
    midpoint=0, limits=c(-3, 3), oob=scales::squish,
    name="NES\n(severe vs\nhealthy)"
  ) +
  facet_grid(rows=vars(collection), scales="free_y", space="free_y") +
  labs(
    title="GSEA: severe IVD degeneration vs healthy",
    subtitle="Normalized enrichment score | * padj < 0.05 | Blue = downregulated in severe",
    x=NULL, y=NULL
  ) +
  theme_classic(base_size=9.5) +
  theme(
    axis.text.x  = element_text(angle=35, hjust=1, size=9),
    axis.text.y  = element_text(size=8),
    strip.text   = element_text(face="bold", size=9),
    strip.background = element_rect(fill="gray92", color=NA),
    plot.title   = element_text(face="bold", size=12),
    plot.subtitle= element_text(size=8.5, color="gray30"),
    legend.position="right",
    panel.spacing= unit(0.3, "lines")
  )

ggsave("/mnt/results/08_pathways/gsea_heatmap_severe_vs_healthy.png",
       p, width=13, height=14, dpi=150)
cat("Saved gsea_heatmap_severe_vs_healthy.png\n")


Total GSEA results: 457 pathway<c3><97>celltype pairs
Significant Hallmarks: 4
[1] "HALLMARK_IL2_STAT5_SIGNALING"               "HALLMARK_EPITHELIAL_MESENCHYMAL_TRANSITION"
[3] "HALLMARK_INFLAMMATORY_RESPONSE"             "HALLMARK_TNFA_SIGNALING_VIA_NFKB"          

Selected Reactome pathways (IVD-relevant, <e2><89><a5>2 CTs): 28
Saved gsea_heatmap_severe_vs_healthy.png

Run PAGA + diffusion pseudotime trajectory on NP chondrocyte subclusters

In [49]:

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('/mnt/results/09_trajectory', exist_ok=True)

# ── Load integrated data and subset to NP lineage ─────────────────────────────
print("Loading integrated AnnData...")
adata = sc.read_h5ad('/workspace/data/processed/adata_annotated.h5ad')

cluster_annotation = {
    '0': 'NP_chondrocyte', '1': 'NP_chondrocyte_HAPLN1', '2': 'NP_stress_response',
    '3': 'NP_degenerative_UPR', '4': 'AF_fibroblast', '5': 'NP_metallothionein',
    '6': 'T_NK_cell', '7': 'Pericyte_SMC', '8': 'Endothelial',
    '9': 'Erythrocyte', '10': 'Macrophage', '11': 'Monocyte_Neutrophil',
}
adata.obs['cell_type'] = adata.obs['leiden_0.5'].map(cluster_annotation)

# Subset to NP lineage (5 NP states)
np_types = ['NP_chondrocyte', 'NP_chondrocyte_HAPLN1', 'NP_stress_response',
            'NP_degenerative_UPR', 'NP_metallothionein']
np_idx = np.where(adata.obs['cell_type'].isin(np_types))[0]
adata_np = adata[np_idx].copy()
print(f"NP lineage subset: {adata_np.shape}")
print(adata_np.obs['cell_type'].value_counts())

# Use Harmony embedding for neighbor graph (already computed)
# Re-run neighbors on NP subset using Harmony PCs
print("\nRecomputing neighbors on NP subset (Harmony PCs)...")
sc.pp.neighbors(adata_np, use_rep='X_pca_harmony', n_neighbors=20, n_pcs=30)

# PAGA
print("Running PAGA...")
sc.tl.paga(adata_np, groups='cell_type')
sc.pl.paga(adata_np, show=False)

# UMAP on NP subset
print("Running UMAP on NP subset...")
sc.tl.umap(adata_np, min_dist=0.4, spread=1.0)

# Diffusion pseudotime — root in NP_chondrocyte (canonical healthy state)
print("Computing diffusion map + pseudotime...")
sc.tl.diffmap(adata_np, n_comps=15)

# Set root: cell in NP_chondrocyte cluster from healthy condition with highest DC1
healthy_np_mask = (adata_np.obs['cell_type'] == 'NP_chondrocyte') & \
                  (adata_np.obs['condition_harmonized'] == 'healthy')
healthy_np_idx = np.where(healthy_np_mask)[0]
# Root = cell with most extreme DC1 value in healthy NP_chondrocyte
dc1 = adata_np.obsm['X_diffmap'][healthy_np_idx, 1]
root_local = healthy_np_idx[np.argmax(dc1)]
adata_np.uns['iroot'] = root_local
print(f"  Root cell index: {root_local} (NP_chondrocyte, healthy)")

sc.tl.dpt(adata_np, n_dcs=10)
print(f"  Pseudotime range: {adata_np.obs['dpt_pseudotime'].min():.3f} – {adata_np.obs['dpt_pseudotime'].max():.3f}")

# Save NP subset
adata_np.write_h5ad('/workspace/data/processed/adata_np_trajectory.h5ad')
print("Saved adata_np_trajectory.h5ad")


Loading integrated AnnData...
NP lineage subset: (140439, 25304)
cell_type
NP_chondrocyte           47120
NP_degenerative_UPR      34810
NP_metallothionein       31502
NP_chondrocyte_HAPLN1    17040
NP_stress_response        9967
Name: count, dtype: int64

Recomputing neighbors on NP subset (Harmony PCs)...
Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 35, in <module>
  File "/opt/conda/lib/python3.11/site-packages/scanpy/neighbors/__init__.py", line 194, in neighbors
    neighbors.compute_neighbors(
  File "/opt/conda/lib/python3.11/site-packages/scanpy/neighbors/__init__.py", line 586, in compute_neighbors
    X = _choose_representation(self._adata, use_rep=use_rep, n_pcs=n_pcs)
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/scanpy/tools/_utils.py", line 47, in _choose_representation
    raise ValueError(msg)
ValueError: Did not find X_pca_harmony in `.obsm.

Check available obsm keys and use correct Harmony embedding name

In [51]:

print("Available obsm keys:", list(adata_np.obsm.keys()))
print("Available uns keys:", list(adata_np.uns.keys())[:10])


Available obsm keys: ['X_harmony', 'X_pca', 'X_umap']
Available uns keys: ['hvg', 'leiden_0.3', 'leiden_0.5', 'leiden_0.8', 'leiden_1.2', 'log1p', 'neighbors', 'rank_genes_0.5', 'umap']


Run PAGA + diffusion pseudotime using correct X_harmony embedding

In [53]:

import scanpy as sc
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("Recomputing neighbors on NP subset (X_harmony)...")
sc.pp.neighbors(adata_np, use_rep='X_harmony', n_neighbors=20, n_pcs=30)

# PAGA
print("Running PAGA...")
sc.tl.paga(adata_np, groups='cell_type')

# UMAP on NP subset
print("Running UMAP on NP subset...")
sc.tl.umap(adata_np, min_dist=0.4, spread=1.0)

# Diffusion map + pseudotime
print("Computing diffusion map...")
sc.tl.diffmap(adata_np, n_comps=15)

# Root: NP_chondrocyte healthy cell with highest DC1
healthy_np_mask = (adata_np.obs['cell_type'] == 'NP_chondrocyte') & \
                  (adata_np.obs['condition_harmonized'] == 'healthy')
healthy_np_idx = np.where(healthy_np_mask)[0]
dc1 = adata_np.obsm['X_diffmap'][healthy_np_idx, 1]
root_local = healthy_np_idx[np.argmax(dc1)]
adata_np.uns['iroot'] = root_local
print(f"  Root cell: index {root_local} (NP_chondrocyte, healthy)")

print("Computing DPT pseudotime...")
sc.tl.dpt(adata_np, n_dcs=10)
print(f"  Pseudotime range: {adata_np.obs['dpt_pseudotime'].min():.3f} – {adata_np.obs['dpt_pseudotime'].max():.3f}")

# Mean pseudotime per cell type
print("\nMean pseudotime per cell type:")
print(adata_np.obs.groupby('cell_type')['dpt_pseudotime'].mean().sort_values().round(3))

adata_np.write_h5ad('/workspace/data/processed/adata_np_trajectory.h5ad')
print("\nSaved adata_np_trajectory.h5ad")


Recomputing neighbors on NP subset (X_harmony)...
Running PAGA...
Running UMAP on NP subset...
Computing diffusion map...
  Root cell: index 11412 (NP_chondrocyte, healthy)
Computing DPT pseudotime...
  Pseudotime range: 0.000 – 1.000

Mean pseudotime per cell type:
cell_type
NP_stress_response       0.072
NP_degenerative_UPR      0.072
NP_chondrocyte           0.080
NP_metallothionein       0.087
NP_chondrocyte_HAPLN1    0.094
Name: dpt_pseudotime, dtype: float32

Saved adata_np_trajectory.h5ad


Generate trajectory figures: PAGA graph, NP UMAP colored by pseudotime/condition/cell type

In [55]:

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

ct_colors = {
    'NP_chondrocyte':        '#2166AC',
    'NP_chondrocyte_HAPLN1': '#4393C3',
    'NP_stress_response':    '#92C5DE',
    'NP_degenerative_UPR':   '#D1E5F0',
    'NP_metallothionein':    '#74ADD1',
}
ct_labels_short = {
    'NP_chondrocyte':        'NP: canonical',
    'NP_chondrocyte_HAPLN1': 'NP: HAPLN1+',
    'NP_stress_response':    'NP: stress',
    'NP_degenerative_UPR':   'NP: degenerative',
    'NP_metallothionein':    'NP: MT-high',
}
cond_colors = {
    'healthy':         '#2166AC',
    'mild_degen':      '#FEE090',
    'moderate_degen':  '#F46D43',
    'severe_degen':    '#A50026',
}

umap = adata_np.obsm['X_umap']
cell_types = adata_np.obs['cell_type'].values
conditions = adata_np.obs['condition_harmonized'].values
pseudotime = adata_np.obs['dpt_pseudotime'].values

fig = plt.figure(figsize=(18, 12))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.3)

# ── Panel A: NP UMAP colored by cell type ─────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor('#F8F8F8')
for ct, color in ct_colors.items():
    mask = cell_types == ct
    ax0.scatter(umap[mask, 0], umap[mask, 1], c=color, s=0.3, alpha=0.5,
                rasterized=True, label=ct_labels_short[ct])
ax0.set_title('NP lineage cell types\n(140,439 cells)', fontsize=10, fontweight='bold')
ax0.set_xlabel('UMAP1', fontsize=9); ax0.set_ylabel('UMAP2', fontsize=9)
handles = [mpatches.Patch(color=ct_colors[ct], label=ct_labels_short[ct]) for ct in ct_colors]
ax0.legend(handles=handles, fontsize=7, loc='upper right', framealpha=0.9)

# ── Panel B: NP UMAP colored by condition ─────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
ax1.set_facecolor('#F8F8F8')
for cond, color in cond_colors.items():
    mask = conditions == cond
    ax1.scatter(umap[mask, 0], umap[mask, 1], c=color, s=0.3, alpha=0.4,
                rasterized=True, label=cond)
ax1.set_title('Degeneration condition', fontsize=10, fontweight='bold')
ax1.set_xlabel('UMAP1', fontsize=9); ax1.set_ylabel('UMAP2', fontsize=9)
handles2 = [mpatches.Patch(color=c, label=l) for l, c in cond_colors.items()]
ax1.legend(handles=handles2, fontsize=7.5, loc='upper right', framealpha=0.9)

# ── Panel C: NP UMAP colored by pseudotime ────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
ax2.set_facecolor('#F8F8F8')
# Plot in pseudotime order so high-PT cells are on top
order = np.argsort(pseudotime)
sc_pt = ax2.scatter(umap[order, 0], umap[order, 1],
                    c=pseudotime[order], cmap='viridis_r',
                    s=0.3, alpha=0.6, rasterized=True)
plt.colorbar(sc_pt, ax=ax2, label='Pseudotime', shrink=0.8)
ax2.set_title('Diffusion pseudotime\n(root: healthy NP canonical)', fontsize=10, fontweight='bold')
ax2.set_xlabel('UMAP1', fontsize=9); ax2.set_ylabel('UMAP2', fontsize=9)

# ── Panel D: PAGA graph ────────────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
sc.pl.paga(adata_np, color=['cell_type'], ax=ax3, show=False,
           node_size_scale=2.5, edge_width_scale=1.5,
           title='PAGA connectivity graph',
           palette=list(ct_colors.values()),
           fontsize=8, frameon=False)

# ── Panel E: Pseudotime distribution per condition ────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
cond_order = ['healthy', 'mild_degen', 'moderate_degen', 'severe_degen']
cond_label_map = {'healthy': 'Healthy', 'mild_degen': 'Mild',
                  'moderate_degen': 'Moderate', 'severe_degen': 'Severe'}
pt_data = pd.DataFrame({'pseudotime': pseudotime, 'condition': conditions})

from scipy.stats import gaussian_kde
x_range = np.linspace(0, 1, 200)
for cond in cond_order:
    sub = pt_data[pt_data['condition'] == cond]['pseudotime'].values
    if len(sub) < 50: continue
    kde = gaussian_kde(sub, bw_method=0.15)
    ax4.plot(x_range, kde(x_range), color=cond_colors[cond],
             linewidth=2, label=cond_label_map[cond])
    ax4.fill_between(x_range, kde(x_range), alpha=0.15, color=cond_colors[cond])

ax4.set_xlabel('Pseudotime', fontsize=10)
ax4.set_ylabel('Density', fontsize=10)
ax4.set_title('Pseudotime distribution\nby degeneration condition', fontsize=10, fontweight='bold')
ax4.legend(fontsize=8.5, framealpha=0.9)
ax4.spines['top'].set_visible(False)
ax4.spines['right'].set_visible(False)

# ── Panel F: Mean pseudotime per cell type × condition ────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
pt_summary = pt_data.copy()
pt_summary['cell_type'] = cell_types
pt_summary['cell_label'] = [ct_labels_short[c] for c in cell_types]

ct_order_plot = ['NP: canonical', 'NP: HAPLN1+', 'NP: stress', 'NP: MT-high', 'NP: degenerative']
means = pt_summary.groupby(['cell_label', 'condition'])['pseudotime'].mean().reset_index()
means['cell_label'] = pd.Categorical(means['cell_label'], categories=ct_order_plot)
means = means.sort_values('cell_label')

for cond in cond_order:
    sub = means[means['condition'] == cond]
    ax5.plot(sub['cell_label'], sub['pseudotime'],
             'o-', color=cond_colors[cond], linewidth=2, markersize=7,
             label=cond_label_map[cond])

ax5.set_xlabel(None)
ax5.set_ylabel('Mean pseudotime', fontsize=10)
ax5.set_title('Mean pseudotime per cell type\nby condition', fontsize=10, fontweight='bold')
ax5.tick_params(axis='x', rotation=30, labelsize=8)
ax5.legend(fontsize=8, framealpha=0.9)
ax5.spines['top'].set_visible(False)
ax5.spines['right'].set_visible(False)

plt.suptitle('NP lineage trajectory: healthy → degenerative states',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('/mnt/results/09_trajectory/np_trajectory_overview.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved np_trajectory_overview.png")


Traceback (most recent call last):
  File "<string>", line 71, in <module>
  File "<string>", line 79, in <module>
  File "/opt/conda/lib/python3.11/site-packages/legacy_api_wrap/__init__.py", line 82, in fn_compatible
    return fn(*args_all, **kw)
           ^^^^^^^^^^^^^^^^^^^
TypeError: paga() got an unexpected keyword argument 'palette'


Diagnose pseudotime quality - check diffusion map variance and pseudotime distributions

In [57]:

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Check diffusion map components
dm = adata_np.obsm['X_diffmap']
print(f"Diffusion map shape: {dm.shape}")
print(f"\nVariance per DC (DC1-DC10):")
for i in range(1, 11):
    print(f"  DC{i}: var={dm[:, i].var():.6f}, range=[{dm[:, i].min():.3f}, {dm[:, i].max():.3f}]")

# Check pseudotime distribution
pt = adata_np.obs['dpt_pseudotime'].values
print(f"\nPseudotime stats:")
print(f"  min={pt.min():.4f}, max={pt.max():.4f}, mean={pt.mean():.4f}, median={np.median(pt):.4f}")
print(f"  % cells with PT < 0.1: {(pt < 0.1).mean()*100:.1f}%")
print(f"  % cells with PT > 0.5: {(pt > 0.5).mean()*100:.1f}%")

# Per cell type distribution
print(f"\nPseudotime per cell type (mean ± std):")
for ct in ['NP_chondrocyte', 'NP_chondrocyte_HAPLN1', 'NP_stress_response',
           'NP_degenerative_UPR', 'NP_metallothionein']:
    mask = adata_np.obs['cell_type'] == ct
    sub = pt[mask]
    print(f"  {ct:<30}: mean={sub.mean():.4f} ± {sub.std():.4f}, "
          f"median={np.median(sub):.4f}, max={sub.max():.4f}")

# Per condition
print(f"\nPseudotime per condition (mean ± std):")
for cond in ['healthy', 'mild_degen', 'moderate_degen', 'severe_degen']:
    mask = adata_np.obs['condition_harmonized'] == cond
    sub = pt[mask]
    if len(sub) > 0:
        print(f"  {cond:<20}: mean={sub.mean():.4f} ± {sub.std():.4f}, n={len(sub)}")


Diffusion map shape: (140439, 15)

Variance per DC (DC1-DC10):
  DC1: var=0.000007, range=[-0.003, 0.012]
  DC2: var=0.000007, range=[-0.009, 0.038]
  DC3: var=0.000007, range=[-0.072, 0.006]
  DC4: var=0.000007, range=[-0.100, 0.023]
  DC5: var=0.000007, range=[-0.030, 0.050]
  DC6: var=0.000007, range=[-0.039, 0.010]
  DC7: var=0.000007, range=[-0.019, 0.068]
  DC8: var=0.000007, range=[-0.033, 0.177]
  DC9: var=0.000007, range=[-0.075, 0.090]
  DC10: var=0.000007, range=[-0.135, 0.093]

Pseudotime stats:
  min=0.0000, max=1.0000, mean=0.0808, median=0.0793
  % cells with PT < 0.1: 86.7%
  % cells with PT > 0.5: 0.1%

Pseudotime per cell type (mean ± std):
  NP_chondrocyte                : mean=0.0801 ± 0.0139, median=0.0806, max=0.1448
  NP_chondrocyte_HAPLN1         : mean=0.0938 ± 0.0135, median=0.0937, max=0.1560
  NP_stress_response            : mean=0.0720 ± 0.0228, median=0.0691, max=0.4636
  NP_degenerative_UPR           : mean=0.0720 ± 0.0136, median=0.0728, max=0.1280
  NP_

Diagnose diffusion map failure and run PAGA-based trajectory on NP subset with leiden subclustering

In [59]:

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# The diffusion map failure is because Harmony over-compressed the embedding.
# Better approach: use the lognorm expression matrix directly for the NP subset,
# rerun PCA → neighbors → PAGA on just the NP cells

print("Rebuilding NP subset from lognorm expression for trajectory...")
print(f"NP subset shape: {adata_np.shape}")
print(f"Available layers: {list(adata_np.layers.keys())}")

# Use lognorm layer for PCA
adata_np.X = adata_np.layers['lognorm'].copy()

# Reselect HVGs within NP subset
sc.pp.highly_variable_genes(adata_np, n_top_genes=3000, batch_key='dataset',
                             flavor='seurat_v3', subset=False)
n_hvg = adata_np.var['highly_variable'].sum()
print(f"HVGs in NP subset: {n_hvg}")

# PCA on NP subset
sc.tl.pca(adata_np, n_comps=30, use_highly_variable=True)
print(f"PCA variance explained (PC1-10): {adata_np.uns['pca']['variance_ratio'][:10].sum()*100:.1f}%")

# Neighbors
sc.pp.neighbors(adata_np, n_neighbors=20, n_pcs=20)

# PAGA on cell types
sc.tl.paga(adata_np, groups='cell_type')

# UMAP
sc.tl.umap(adata_np, min_dist=0.3)

# Diffusion map + DPT
sc.tl.diffmap(adata_np, n_comps=15)

dm = adata_np.obsm['X_diffmap']
print(f"\nDiffusion map variance per DC (DC1-DC5):")
for i in range(1, 6):
    print(f"  DC{i}: var={dm[:, i].var():.6f}, range=[{dm[:, i].min():.4f}, {dm[:, i].max():.4f}]")

# Root: healthy NP_chondrocyte cell with max DC1
healthy_np_mask = (adata_np.obs['cell_type'] == 'NP_chondrocyte') & \
                  (adata_np.obs['condition_harmonized'] == 'healthy')
healthy_np_idx = np.where(healthy_np_mask)[0]
dc1 = dm[healthy_np_idx, 1]
root_local = healthy_np_idx[np.argmax(dc1)]
adata_np.uns['iroot'] = root_local

sc.tl.dpt(adata_np, n_dcs=10)
pt = adata_np.obs['dpt_pseudotime'].values
print(f"\nPseudotime: min={pt.min():.4f}, max={pt.max():.4f}, mean={pt.mean():.4f}")
print(f"  % cells PT < 0.1: {(pt < 0.1).mean()*100:.1f}%")
print(f"  % cells PT > 0.5: {(pt > 0.5).mean()*100:.1f}%")

print("\nMean pseudotime per cell type:")
print(adata_np.obs.groupby('cell_type')['dpt_pseudotime'].mean().sort_values().round(4))
print("\nMean pseudotime per condition:")
print(adata_np.obs.groupby('condition_harmonized')['dpt_pseudotime'].mean().sort_values().round(4))


Rebuilding NP subset from lognorm expression for trajectory...
NP subset shape: (140439, 25304)
Available layers: ['counts', 'lognorm']
HVGs in NP subset: 3000
PCA variance explained (PC1-10): 39.4%

Diffusion map variance per DC (DC1-DC5):
  DC1: var=0.000007, range=[-0.0123, 0.0059]
  DC2: var=0.000007, range=[-0.0100, 0.0018]
  DC3: var=0.000007, range=[-0.0086, 0.0046]
  DC4: var=0.000007, range=[-0.0152, 0.0029]
  DC5: var=0.000007, range=[-0.0130, 0.0148]

Pseudotime: min=0.0000, max=1.0000, mean=0.2218
  % cells PT < 0.1: 22.4%
  % cells PT > 0.5: 3.7%

Mean pseudotime per cell type:
cell_type
NP_metallothionein       0.1575
NP_stress_response       0.2216
NP_chondrocyte           0.2227
NP_chondrocyte_HAPLN1    0.2469
NP_degenerative_UPR      0.2663
Name: dpt_pseudotime, dtype: float32

Mean pseudotime per condition:
condition_harmonized
healthy           0.2077
moderate_degen    0.2172
severe_degen      0.2682
mild_degen        0.2707
Name: dpt_pseudotime, dtype: float32


Assess DPT quality and decide whether to proceed or report limitation

In [61]:

import numpy as np
import pandas as pd
from scipy.stats import kruskal, spearmanr

pt = adata_np.obs['dpt_pseudotime'].values
cell_types_np = adata_np.obs['cell_type'].values
conditions = adata_np.obs['condition_harmonized'].values

# 1. Does pseudotime separate cell types? (Kruskal-Wallis)
groups = [pt[cell_types_np == ct] for ct in 
          ['NP_chondrocyte','NP_chondrocyte_HAPLN1','NP_stress_response',
           'NP_degenerative_UPR','NP_metallothionein']]
H, p_kw = kruskal(*groups)
print(f"Kruskal-Wallis across cell types: H={H:.1f}, p={p_kw:.2e}")

# 2. Does pseudotime correlate with degeneration severity?
cond_numeric = {'healthy': 0, 'mild_degen': 1, 'moderate_degen': 2, 'severe_degen': 3}
cond_num = np.array([cond_numeric.get(c, np.nan) for c in conditions])
valid = ~np.isnan(cond_num)
rho, p_corr = spearmanr(pt[valid], cond_num[valid])
print(f"Spearman correlation PT vs degeneration grade: rho={rho:.4f}, p={p_corr:.2e}")

# 3. Check DC variance — is it truly flat?
dm = adata_np.obsm['X_diffmap']
print(f"\nDC variances (should differ if trajectory exists):")
for i in range(1, 8):
    print(f"  DC{i}: {dm[:, i].var():.2e}")

# 4. Check if PAGA connectivity is meaningful
paga_conn = adata_np.uns['paga']['connectivities'].toarray()
ct_names = adata_np.obs['cell_type'].cat.categories.tolist()
print(f"\nPAGA connectivity matrix ({len(ct_names)} cell types):")
conn_df = pd.DataFrame(paga_conn, index=ct_names, columns=ct_names)
print(conn_df.round(3).to_string())


Kruskal-Wallis across cell types: H=12486.7, p=0.00e+00
Spearman correlation PT vs degeneration grade: rho=0.2352, p=0.00e+00

DC variances (should differ if trajectory exists):
  DC1: 7.12e-06
  DC2: 7.12e-06
  DC3: 7.12e-06
  DC4: 7.12e-06
  DC5: 7.12e-06
  DC6: 7.12e-06
  DC7: 7.12e-06

PAGA connectivity matrix (5 cell types):
                       NP_chondrocyte  NP_chondrocyte_HAPLN1  NP_stress_response  NP_degenerative_UPR  NP_metallothionein
NP_chondrocyte                  0.000                  0.645               0.522                0.424               0.291
NP_chondrocyte_HAPLN1           0.645                  0.000               0.366                0.591               0.197
NP_stress_response              0.522                  0.366               0.000                0.761               0.505
NP_degenerative_UPR             0.424                  0.591               0.761                0.000               0.464
NP_metallothionein              0.291                  0.1

Generate trajectory figure using PAGA connectivity + NP UMAP + condition shifts (no unreliable DPT)

In [63]:

import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('/mnt/results/09_trajectory', exist_ok=True)

ct_colors = {
    'NP_chondrocyte':        '#2166AC',
    'NP_chondrocyte_HAPLN1': '#4393C3',
    'NP_stress_response':    '#92C5DE',
    'NP_degenerative_UPR':   '#D1E5F0',
    'NP_metallothionein':    '#74ADD1',
}
ct_short = {
    'NP_chondrocyte':        'NP: canonical',
    'NP_chondrocyte_HAPLN1': 'NP: HAPLN1+',
    'NP_stress_response':    'NP: stress',
    'NP_degenerative_UPR':   'NP: degenerative',
    'NP_metallothionein':    'NP: MT-high',
}
cond_colors = {
    'healthy':        '#2166AC',
    'mild_degen':     '#FEE090',
    'moderate_degen': '#F46D43',
    'severe_degen':   '#A50026',
}
cond_labels = {'healthy':'Healthy','mild_degen':'Mild',
               'moderate_degen':'Moderate','severe_degen':'Severe'}

umap   = adata_np.obsm['X_umap']
ct_arr = adata_np.obs['cell_type'].values
cond   = adata_np.obs['condition_harmonized'].values
pt     = adata_np.obs['dpt_pseudotime'].values

fig = plt.figure(figsize=(18, 11))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

# ── A: NP UMAP by cell type ───────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0, 0])
ax0.set_facecolor('#F8F8F8')
for ct, color in ct_colors.items():
    m = ct_arr == ct
    ax0.scatter(umap[m,0], umap[m,1], c=color, s=0.3, alpha=0.5, rasterized=True)
ax0.set_title('NP lineage cell types\n(140,439 cells)', fontsize=10, fontweight='bold')
ax0.set_xlabel('UMAP1', fontsize=9); ax0.set_ylabel('UMAP2', fontsize=9)
handles = [mpatches.Patch(color=ct_colors[ct], label=ct_short[ct]) for ct in ct_colors]
ax0.legend(handles=handles, fontsize=7.5, loc='upper right', framealpha=0.9)
ax0.tick_params(labelsize=8)

# ── B: NP UMAP by condition ───────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 1])
ax1.set_facecolor('#F8F8F8')
for c, color in cond_colors.items():
    m = cond == c
    ax1.scatter(umap[m,0], umap[m,1], c=color, s=0.3, alpha=0.4, rasterized=True)
ax1.set_title('Degeneration condition', fontsize=10, fontweight='bold')
ax1.set_xlabel('UMAP1', fontsize=9); ax1.set_ylabel('UMAP2', fontsize=9)
handles2 = [mpatches.Patch(color=c, label=cond_labels[l]) for l,c in cond_colors.items()]
ax1.legend(handles=handles2, fontsize=8, loc='upper right', framealpha=0.9)
ax1.tick_params(labelsize=8)

# ── C: PAGA connectivity heatmap ─────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 2])
paga_conn = adata_np.uns['paga']['connectivities'].toarray()
ct_names  = list(adata_np.obs['cell_type'].cat.categories)
ct_short_list = [ct_short[c] for c in ct_names]

im = ax2.imshow(paga_conn, cmap='Reds', vmin=0, vmax=1, aspect='auto')
ax2.set_xticks(range(len(ct_names)))
ax2.set_yticks(range(len(ct_names)))
ax2.set_xticklabels(ct_short_list, rotation=35, ha='right', fontsize=8)
ax2.set_yticklabels(ct_short_list, fontsize=8)
for i in range(len(ct_names)):
    for j in range(len(ct_names)):
        v = paga_conn[i, j]
        if v > 0.05:
            ax2.text(j, i, f'{v:.2f}', ha='center', va='center',
                     fontsize=7.5, color='white' if v > 0.5 else 'black')
plt.colorbar(im, ax=ax2, shrink=0.8, label='PAGA connectivity')
ax2.set_title('PAGA connectivity\nbetween NP states', fontsize=10, fontweight='bold')

# ── D: Cell type composition shift across conditions ─────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
obs_np = pd.DataFrame({'cell_type': ct_arr, 'condition': cond})
comp = obs_np.groupby(['condition','cell_type']).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100
comp_pct = comp_pct.reindex(['healthy','mild_degen','moderate_degen','severe_degen'])

bottom = np.zeros(4)
x = np.arange(4)
for ct in ct_colors:
    if ct in comp_pct.columns:
        vals = comp_pct[ct].values
        ax3.bar(x, vals, bottom=bottom, color=ct_colors[ct],
                label=ct_short[ct], width=0.65, edgecolor='white', linewidth=0.3)
        bottom += vals
ax3.set_xticks(x)
ax3.set_xticklabels(['Healthy','Mild','Moderate','Severe'], fontsize=9)
ax3.set_ylabel('% of NP cells', fontsize=10)
ax3.set_title('NP state composition\nby degeneration grade', fontsize=10, fontweight='bold')
ax3.yaxis.set_major_formatter(plt.FuncFormatter(lambda y,_: f'{y:.0f}%'))
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
handles3 = [mpatches.Patch(color=ct_colors[ct], label=ct_short[ct]) for ct in ct_colors]
ax3.legend(handles=handles3, fontsize=7, loc='upper right', framealpha=0.9)

# ── E: Pseudotime density per condition (with caveat note) ───────────────────
ax4 = fig.add_subplot(gs[1, 1])
x_range = np.linspace(0, 0.6, 200)
for c in ['healthy','mild_degen','moderate_degen','severe_degen']:
    sub = pt[cond == c]
    if len(sub) < 50: continue
    kde = gaussian_kde(sub, bw_method=0.2)
    ax4.plot(x_range, kde(x_range), color=cond_colors[c],
             linewidth=2, label=cond_labels[c])
    ax4.fill_between(x_range, kde(x_range), alpha=0.12, color=cond_colors[c])
ax4.set_xlabel('Pseudotime (DPT)', fontsize=10)
ax4.set_ylabel('Density', fontsize=10)
ax4.set_title('Pseudotime distribution\nby condition (ρ=0.24, p<10⁻³⁰⁰)', fontsize=10, fontweight='bold')
ax4.legend(fontsize=8.5, framealpha=0.9)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)
ax4.text(0.97, 0.97, '⚠ DPT weakly resolved\n(see Methods)', transform=ax4.transAxes,
         fontsize=7, ha='right', va='top', color='gray',
         bbox=dict(boxstyle='round,pad=0.3', facecolor='lightyellow', alpha=0.8))

# ── F: NP state % change: severe vs healthy ──────────────────────────────────
ax5 = fig.add_subplot(gs[1, 2])
pct_healthy = comp_pct.loc['healthy']
pct_severe  = comp_pct.loc['severe_degen']
delta = (pct_severe - pct_healthy).sort_values()
colors_bar = [ct_colors.get(ct, '#AAAAAA') for ct in delta.index]
labels_bar  = [ct_short.get(ct, ct) for ct in delta.index]
bars = ax5.barh(labels_bar, delta.values, color=colors_bar, edgecolor='white', height=0.6)
ax5.axvline(0, color='black', linewidth=0.8)
for bar, val in zip(bars, delta.values):
    ax5.text(val + (0.3 if val >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
             f'{val:+.1f}%', va='center', ha='left' if val >= 0 else 'right', fontsize=8)
ax5.set_xlabel('Δ proportion (severe − healthy, %)', fontsize=9)
ax5.set_title('NP state shift:\nsevere vs healthy', fontsize=10, fontweight='bold')
ax5.spines['top'].set_visible(False); ax5.spines['right'].set_visible(False)

plt.suptitle('NP lineage trajectory & state transitions: healthy → severe degeneration',
             fontsize=12, fontweight='bold', y=1.01)

plt.savefig('/mnt/results/09_trajectory/np_trajectory_overview.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved np_trajectory_overview.png")


Saved np_trajectory_overview.png


Verify all trajectory data comes from real adata_np object, not simulated values

In [65]:

import numpy as np
import pandas as pd

# Cross-check all data used in the trajectory figure against the real adata_np

print("=== DATA INTEGRITY CHECK: Trajectory Figure ===\n")

# 1. Cell counts
print("1. NP cell type counts (from adata_np.obs):")
ct_counts = adata_np.obs['cell_type'].value_counts()
print(ct_counts.to_string())
print(f"   Total: {ct_counts.sum():,} (expected 140,439)")
assert ct_counts.sum() == 140439, "Cell count mismatch!"
print("   ✓ Matches\n")

# 2. PAGA connectivity — verify it's from uns, not hardcoded
paga_conn = adata_np.uns['paga']['connectivities'].toarray()
print("2. PAGA connectivity matrix source: adata_np.uns['paga']['connectivities']")
print(f"   Shape: {paga_conn.shape}, dtype: {paga_conn.dtype}")
print(f"   Max connectivity: {paga_conn.max():.3f} (NP_stress ↔ NP_degenerative)")
print(f"   Min off-diagonal: {paga_conn[paga_conn > 0].min():.3f}")
print("   ✓ Real PAGA output\n")

# 3. Composition data — verify against raw obs
obs_np = adata_np.obs[['cell_type','condition_harmonized']].copy()
comp = obs_np.groupby(['condition_harmonized','cell_type']).size().unstack(fill_value=0)
comp_pct = comp.div(comp.sum(axis=1), axis=0) * 100

print("3. NP composition per condition (%):")
print(comp_pct.round(1).to_string())
print()

# Verify the delta values shown in panel F
pct_healthy = comp_pct.loc['healthy']
pct_severe  = comp_pct.loc['severe_degen']
delta = (pct_severe - pct_healthy).sort_values()
print("   Delta (severe - healthy):")
for ct, d in delta.items():
    print(f"     {ct}: {d:+.1f}%")
print("   ✓ Computed from real obs data\n")

# 4. Pseudotime — verify it's from adata_np.obs
pt = adata_np.obs['dpt_pseudotime'].values
print("4. Pseudotime source: adata_np.obs['dpt_pseudotime']")
print(f"   n={len(pt):,}, min={pt.min():.4f}, max={pt.max():.4f}, mean={pt.mean():.4f}")
print(f"   ⚠ DPT weakly resolved (all DC variances = 7.12e-06) — labeled in figure")
print(f"   Spearman rho with degeneration grade = 0.24 (p<1e-300) — weak but real signal")


=== DATA INTEGRITY CHECK: Trajectory Figure ===

1. NP cell type counts (from adata_np.obs):
cell_type
NP_chondrocyte           47120
NP_degenerative_UPR      34810
NP_metallothionein       31502
NP_chondrocyte_HAPLN1    17040
NP_stress_response        9967
   Total: 140,439 (expected 140,439)
   ✓ Matches

2. PAGA connectivity matrix source: adata_np.uns['paga']['connectivities']
   Shape: (5, 5), dtype: float64
   Max connectivity: 0.761 (NP_stress ↔ NP_degenerative)
   Min off-diagonal: 0.197
   ✓ Real PAGA output

3. NP composition per condition (%):
cell_type             NP_chondrocyte  NP_chondrocyte_HAPLN1  NP_stress_response  NP_degenerative_UPR  NP_metallothionein
condition_harmonized                                                                                                    
healthy                         35.0                   10.2                 6.5                 22.0                26.3
mild_degen                      48.0                   10.2                 

Run LIANA cell-cell communication analysis on healthy vs severe degeneration

In [67]:

import subprocess
result = subprocess.run(['pip', 'install', 'liana', '-q'], capture_output=True, text=True)
print(result.stdout[-200:] if result.stdout else "installed")
print(result.stderr[-200:] if result.stderr else "")

import liana
print(f"LIANA version: {liana.__version__}")


installed
nusable. It is recommended to use a virtual environment instead: https://pip.pypa.io/warnings/venv. Use the --root-user-action option if you know what you are doing and want to suppress this warning.

LIANA version: 1.7.1


Prepare LIANA input: subset to healthy and severe, use all annotated cell types

In [69]:

import scanpy as sc
import numpy as np
import pandas as pd
import liana
import warnings
warnings.filterwarnings('ignore')
import os
os.makedirs('/mnt/results/10_cellchat', exist_ok=True)

# Load full annotated data
print("Loading full annotated AnnData...")
adata_full = sc.read_h5ad('/workspace/data/processed/adata_annotated.h5ad')

cluster_annotation = {
    '0': 'NP_chondrocyte', '1': 'NP_chondrocyte_HAPLN1', '2': 'NP_stress_response',
    '3': 'NP_degenerative_UPR', '4': 'AF_fibroblast', '5': 'NP_metallothionein',
    '6': 'T_NK_cell', '7': 'Pericyte_SMC', '8': 'Endothelial',
    '9': 'Erythrocyte', '10': 'Macrophage', '11': 'Monocyte_Neutrophil',
}
adata_full.obs['cell_type'] = adata_full.obs['leiden_0.5'].map(cluster_annotation)

# Subset to healthy and severe only (clearest contrast)
# Exclude erythrocytes (no signaling relevance) and monocyte/neutrophil (batch-specific)
exclude_types = ['Erythrocyte', 'Monocyte_Neutrophil']
keep_mask = (
    adata_full.obs['condition_harmonized'].isin(['healthy', 'severe_degen']) &
    ~adata_full.obs['cell_type'].isin(exclude_types)
)
adata_liana = adata_full[keep_mask].copy()
print(f"LIANA subset: {adata_liana.shape}")
print(f"Cell types: {adata_liana.obs['cell_type'].value_counts().to_dict()}")
print(f"Conditions: {adata_liana.obs['condition_harmonized'].value_counts().to_dict()}")

# LIANA needs lognorm counts in X
adata_liana.X = adata_liana.layers['lognorm'].copy()

# Check available resources
print(f"\nAvailable LIANA resources: {liana.resource.show_resources()[:8]}")


Loading full annotated AnnData...
LIANA subset: (119428, 25304)
Cell types: {'NP_chondrocyte': 34280, 'NP_metallothionein': 24558, 'NP_degenerative_UPR': 22204, 'AF_fibroblast': 11818, 'NP_chondrocyte_HAPLN1': 11481, 'NP_stress_response': 6944, 'Pericyte_SMC': 3691, 'Endothelial': 3120, 'Macrophage': 934, 'T_NK_cell': 398}
Conditions: {'healthy': 99883, 'severe_degen': 19545}

Available LIANA resources: ['baccin2019', 'cellcall', 'cellchatdb', 'cellinker', 'cellphonedb', 'celltalkdb', 'connectomedb2020', 'consensus']


Run LIANA rank_aggregate on healthy and severe subsets separately

In [71]:

import liana
from liana.method import rank_aggregate
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Run LIANA separately on healthy and severe subsets
results = {}

for cond in ['healthy', 'severe_degen']:
    print(f"\n=== Running LIANA: {cond} ===")
    sub = adata_liana[adata_liana.obs['condition_harmonized'] == cond].copy()
    
    # Need minimum cells per cell type — check
    ct_counts = sub.obs['cell_type'].value_counts()
    valid_cts = ct_counts[ct_counts >= 30].index.tolist()
    sub = sub[sub.obs['cell_type'].isin(valid_cts)].copy()
    print(f"  Cells: {sub.shape[0]}, Cell types: {len(valid_cts)}")
    
    # Set cell type as categorical
    sub.obs['cell_type'] = pd.Categorical(sub.obs['cell_type'])
    
    # Run rank_aggregate with consensus resource
    rank_aggregate(
        sub,
        groupby='cell_type',
        resource_name='consensus',
        expr_prop=0.1,       # min fraction of cells expressing ligand/receptor
        min_cells=30,
        use_raw=False,
        verbose=False,
        n_perms=None,        # skip permutation for speed
    )
    
    res_df = sub.uns['liana_res'].copy()
    res_df['condition'] = cond
    results[cond] = res_df
    print(f"  LR pairs computed: {len(res_df):,}")
    print(f"  Top 5 by magnitude_rank:")
    print(res_df.nsmallest(5, 'magnitude_rank')[
        ['source','target','ligand_complex','receptor_complex','magnitude_rank']
    ].to_string(index=False))

print("\nLIANA complete.")



=== Running LIANA: healthy ===
  Cells: 99883, Cell types: 10
  LR pairs computed: 24,079
  Top 5 by magnitude_rank:
               source                target ligand_complex receptor_complex  magnitude_rank
NP_chondrocyte_HAPLN1 NP_chondrocyte_HAPLN1          TIMP1             CD63    1.552221e-08
        AF_fibroblast NP_chondrocyte_HAPLN1          TIMP1             CD63    6.208713e-08
       NP_chondrocyte NP_chondrocyte_HAPLN1          TIMP1             CD63    2.483348e-07
NP_chondrocyte_HAPLN1        NP_chondrocyte          TIMP1             CD63    3.880123e-07
        AF_fibroblast        NP_chondrocyte          TIMP1             CD63    7.604620e-07

=== Running LIANA: severe_degen ===
  Cells: 19545, Cell types: 10
  LR pairs computed: 35,831
  Top 5 by magnitude_rank:
               source                target ligand_complex receptor_complex  magnitude_rank
NP_chondrocyte_HAPLN1           Endothelial            FN1            ITGA6    7.009977e-09
NP_chondrocyte_HAPLN1  

Compare healthy vs severe LIANA results and identify differentially active LR pairs

In [73]:

import pandas as pd
import numpy as np

# Merge healthy and severe results on source-target-LR pair
healthy_df = results['healthy'].copy()
severe_df  = results['severe_degen'].copy()

# Use magnitude_rank as the score (lower = stronger interaction)
# Convert to -log10 for easier comparison
healthy_df['score_h'] = -np.log10(healthy_df['magnitude_rank'].clip(1e-300))
severe_df['score_s']  = -np.log10(severe_df['magnitude_rank'].clip(1e-300))

# Merge on interaction key
key_cols = ['source', 'target', 'ligand_complex', 'receptor_complex']
merged = pd.merge(
    healthy_df[key_cols + ['score_h']],
    severe_df[key_cols  + ['score_s']],
    on=key_cols, how='outer'
).fillna(0)

merged['delta_score'] = merged['score_s'] - merged['score_h']
merged['interaction'] = merged['ligand_complex'] + ' → ' + merged['receptor_complex']
merged['pair'] = merged['source'] + ' → ' + merged['target']

print(f"Total LR pairs in merged: {len(merged):,}")
print(f"Pairs only in healthy: {(merged['score_s']==0).sum():,}")
print(f"Pairs only in severe:  {(merged['score_h']==0).sum():,}")
print(f"Pairs in both:         {((merged['score_h']>0) & (merged['score_s']>0)).sum():,}")

# Top gained interactions in severe
print("\n=== Top 15 GAINED interactions in severe degeneration ===")
gained = merged.nlargest(15, 'delta_score')[
    ['source','target','ligand_complex','receptor_complex','score_h','score_s','delta_score']
]
print(gained.to_string(index=False))

# Top lost interactions in severe
print("\n=== Top 15 LOST interactions in severe degeneration ===")
lost = merged.nsmallest(15, 'delta_score')[
    ['source','target','ligand_complex','receptor_complex','score_h','score_s','delta_score']
]
print(lost.to_string(index=False))

# Save
merged.to_csv('/mnt/results/10_cellchat/liana_healthy_vs_severe_all.csv', index=False)
gained.to_csv('/mnt/results/10_cellchat/liana_gained_severe.csv', index=False)
lost.to_csv('/mnt/results/10_cellchat/liana_lost_severe.csv', index=False)
print("\nSaved LIANA comparison tables.")


Total LR pairs in merged: 39,530
Pairs only in healthy: 11,632
Pairs only in severe:  20,531
Pairs in both:         14,140

=== Top 15 GAINED interactions in severe degeneration ===
               source                target ligand_complex receptor_complex  score_h  score_s  delta_score
NP_chondrocyte_HAPLN1           Endothelial            FN1            ITGA6 3.854731 8.154283     4.299552
NP_chondrocyte_HAPLN1            Macrophage            FN1            C5AR1 0.000000 4.088301     4.088301
        AF_fibroblast           Endothelial         COL1A2             CD93 0.000000 3.934140     3.934140
NP_chondrocyte_HAPLN1            Macrophage            FN1             CD44 2.830055 6.756376     3.926321
NP_chondrocyte_HAPLN1             T_NK_cell            FN1             CD44 1.841155 5.510006     3.668852
NP_chondrocyte_HAPLN1 NP_chondrocyte_HAPLN1           IL16              CD4 0.000000 3.600029     3.600029
  NP_degenerative_UPR            Macrophage            FN1           

Generate LIANA cell-cell communication chord/bubble plots for healthy vs severe

In [75]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

merged = pd.read_csv('/mnt/results/10_cellchat/liana_healthy_vs_severe_all.csv')

ct_colors = {
    'NP_chondrocyte':        '#2166AC',
    'NP_chondrocyte_HAPLN1': '#4393C3',
    'NP_stress_response':    '#92C5DE',
    'NP_degenerative_UPR':   '#D1E5F0',
    'NP_metallothionein':    '#74ADD1',
    'AF_fibroblast':         '#D6604D',
    'Pericyte_SMC':          '#F4A582',
    'Endothelial':           '#FDAE61',
    'Macrophage':            '#4DAC26',
    'T_NK_cell':             '#762A83',
}
ct_short = {
    'NP_chondrocyte':        'NP: canonical',
    'NP_chondrocyte_HAPLN1': 'NP: HAPLN1+',
    'NP_stress_response':    'NP: stress',
    'NP_degenerative_UPR':   'NP: degenerative',
    'NP_metallothionein':    'NP: MT-high',
    'AF_fibroblast':         'AF fibroblast',
    'Pericyte_SMC':          'Pericyte/SMC',
    'Endothelial':           'Endothelial',
    'Macrophage':            'Macrophage',
    'T_NK_cell':             'T/NK cell',
}

fig = plt.figure(figsize=(20, 14))
fig.patch.set_facecolor('white')
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

# ── Panel A: Total interaction strength per cell type pair (healthy) ──────────
def make_interaction_matrix(df, score_col, cell_types):
    mat = pd.DataFrame(0.0, index=cell_types, columns=cell_types)
    for _, row in df.iterrows():
        s, t = row['source'], row['target']
        if s in cell_types and t in cell_types:
            mat.loc[s, t] += row[score_col]
    return mat

all_cts = list(ct_colors.keys())

# Aggregate total score per source-target pair
h_agg = merged.groupby(['source','target'])['score_h'].sum().reset_index()
s_agg = merged.groupby(['source','target'])['score_s'].sum().reset_index()

mat_h = make_interaction_matrix(h_agg, 'score_h', all_cts)
mat_s = make_interaction_matrix(s_agg, 'score_s', all_cts)
mat_delta = mat_s - mat_h

short_labels = [ct_short[ct] for ct in all_cts]

for ax_idx, (mat, title, cmap, vcenter) in enumerate([
    (mat_h,     'Interaction strength\n(Healthy)',          'Blues',    None),
    (mat_s,     'Interaction strength\n(Severe degen)',     'Reds',     None),
    (mat_delta, 'Δ Interaction strength\n(Severe−Healthy)', 'RdBu_r',   0),
]):
    ax = fig.add_subplot(gs[0, ax_idx])
    vals = mat.values
    if vcenter is not None:
        vmax = np.abs(vals).max()
        im = ax.imshow(vals, cmap=cmap, vmin=-vmax, vmax=vmax, aspect='auto')
    else:
        im = ax.imshow(vals, cmap=cmap, aspect='auto')
    plt.colorbar(im, ax=ax, shrink=0.75, label='Σ -log10(rank)')
    ax.set_xticks(range(len(all_cts)))
    ax.set_yticks(range(len(all_cts)))
    ax.set_xticklabels(short_labels, rotation=45, ha='right', fontsize=7)
    ax.set_yticklabels(short_labels, fontsize=7)
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xlabel('Target', fontsize=8)
    ax.set_ylabel('Source', fontsize=8)

# ── Panel D: Top gained LR pairs (bubble plot) ───────────────────────────────
ax3 = fig.add_subplot(gs[1, 0:2])

# Top 20 gained + top 20 lost by delta_score
top_gained = merged.nlargest(20, 'delta_score').copy()
top_lost   = merged.nsmallest(20, 'delta_score').copy()
plot_df    = pd.concat([top_gained, top_lost])

plot_df['lr_label']   = plot_df['ligand_complex'] + '→' + plot_df['receptor_complex']
plot_df['pair_label'] = plot_df['source'].map(ct_short) + '\n→ ' + plot_df['target'].map(ct_short)
plot_df['direction']  = np.where(plot_df['delta_score'] > 0, 'Gained in severe', 'Lost in severe')
plot_df = plot_df.sort_values('delta_score')

colors_dir = {'Gained in severe': '#D6604D', 'Lost in severe': '#4393C3'}
y_pos = np.arange(len(plot_df))

ax3.barh(y_pos, plot_df['delta_score'].values,
         color=[colors_dir[d] for d in plot_df['direction']],
         height=0.7, edgecolor='white', linewidth=0.3)
ax3.set_yticks(y_pos)
ax3.set_yticklabels([f"{r['lr_label']}  ({r['pair_label'].replace(chr(10),' ')})"
                     for _, r in plot_df.iterrows()], fontsize=7.5)
ax3.axvline(0, color='black', linewidth=0.8)
ax3.set_xlabel('Δ interaction score (severe − healthy)', fontsize=10)
ax3.set_title('Top gained and lost ligand-receptor interactions\n(severe vs healthy degeneration)',
              fontsize=10, fontweight='bold')
ax3.spines['top'].set_visible(False); ax3.spines['right'].set_visible(False)
handles_d = [mpatches.Patch(color=c, label=l) for l, c in colors_dir.items()]
ax3.legend(handles=handles_d, fontsize=9, loc='lower right')

# ── Panel F: Key LR pairs across conditions ───────────────────────────────────
ax4 = fig.add_subplot(gs[1, 2])

# Focus on TIMP1→CD63 (lost) and FN1→CD44 (gained) across all source-target pairs
key_lrs = [('TIMP1','CD63'), ('FN1','CD44'), ('FN1','C5AR1'), ('COL1A2','CD93')]
lr_summary = []
for lig, rec in key_lrs:
    sub = merged[(merged['ligand_complex']==lig) & (merged['receptor_complex']==rec)]
    if len(sub) == 0: continue
    lr_summary.append({
        'LR': f'{lig}→{rec}',
        'Healthy': sub['score_h'].sum(),
        'Severe':  sub['score_s'].sum(),
    })

lr_df = pd.DataFrame(lr_summary)
x = np.arange(len(lr_df))
w = 0.35
ax4.bar(x - w/2, lr_df['Healthy'], width=w, color='#2166AC', label='Healthy', alpha=0.85)
ax4.bar(x + w/2, lr_df['Severe'],  width=w, color='#A50026', label='Severe',  alpha=0.85)
ax4.set_xticks(x)
ax4.set_xticklabels(lr_df['LR'], fontsize=9, rotation=20, ha='right')
ax4.set_ylabel('Total interaction score\n(Σ -log10 rank across all pairs)', fontsize=8.5)
ax4.set_title('Key LR pair activity:\nhealthy vs severe', fontsize=10, fontweight='bold')
ax4.legend(fontsize=9)
ax4.spines['top'].set_visible(False); ax4.spines['right'].set_visible(False)

plt.suptitle('Cell-cell communication: healthy vs severe IVD degeneration (LIANA)',
             fontsize=13, fontweight='bold', y=1.01)

plt.savefig('/mnt/results/10_cellchat/liana_communication_overview.png',
            dpi=150, bbox_inches='tight')
plt.close()
print("Saved liana_communication_overview.png")


Saved liana_communication_overview.png
